# Job Listing Prioritization for Career Changers: EDA/SDA 1 (Exploratory and Statistical Data Analysis)

PredictLeads × TripleTen externship · Jacob Colombo

**The project:** for a given career changer (anyone switching from their current background to a different role or field), suggest the computer and mathematical roles their skills transfer to (**role discovery**), hide the listings they can't get, and rank the rest by which to apply to first, and why. The **retraining compass** is the foundation: prior career → training → first tech job → that role's 2026 openings, pay, closing speed and expected growth. **This notebook** explores the data and decides where to go in depth: which factors, which time information, and which segment.

### Where this notebook fits

| # | Notebook | Question | Hands off |
|:-:|:--|:--|:--|
| **1** | **EDA_SDA_1 (you are here)** | **What's in the data, and what's the scope?** | **Final table: English US listings, April–September 2026 (late March kept, flagged), all groups and candidate factors** |
| 2 | EDA_SDA_2 | Which listings do we study? Additional cleaning, ghost listings, non-text checks, the scope (computer and mathematical, by role bucket), blockers hidden, the compass's 2026 side | Scope table, non-text features |
| 3 | EDA_SDA_3 | What do the descriptions say? Text features, validated blockers, skills mapped to O*NET (no matching models) | Scope table + text features |
| 4 | EDA_SDA_4 | Which features hold up? Statistics | Modeling-ready table |
| 5 | TimeSeries | When do listings appear and close? | Timing features |
| 6 | ML_Fit | Which roles and listings fit a career changer? Keyword → TF-IDF → BGE → RRF, skills-based matching, a trained layer on real career moves | Fit scores, role discovery, the compass |
| 7 | ML_Rank | How should listings be ranked? The listing timing model, combined with fit into the priority queue | The priority queue algorithm |
| side | Profiles (frozen) | Who changed careers into or within tech or math, and how? (public resume dataset; three career-changer pools) | The compass's past side (path tables) and the modeling table for ML |

Each notebook hands its final table to the next. The TripleTen course templates are the baseline for every notebook, taken further with advanced methods.

---

## Executive summary

| | |
|:--|:--|
| **Data** | The full job-openings file: 13,415,388 listings, of which 13,017,464 are English (or unlabeled) and in the US (or no country), first seen 2026-03-30 to 2026-09-30 (PredictLeads snapshot of 2026-09-30). Sections 2–5 use the first 200,000 documents |
| **After cleaning** | 195,801 original listings (no rows removed; 4,199 reposts flagged), 42,399 still open |
| **Final table** | 184,889 English-language listings from 47,252 companies, first seen April–September 2026 plus the data's first two days in late March (flagged `partial_month`); US, plus flagged listings with no country; 41,984 still open |

### Key findings

| Finding | Evidence | What it means for a career changer |
|:--|:--|:--|
| **Company hiring volume is the strongest signal** | Listings at companies with 20+ openings close in a median **13 days** vs. **30** for one-off listings, counting only each company's earlier listings (the way the model sees it; 14 vs. 32 over the whole six months). Part of the gap is crawl timing: like-for-like, measured from the employer's posting date, it's about 1.4–1.5× (EDA_SDA_2 3.3) | High-volume employers' listings need action first |
| **The window is short** | **28%** of listings are gone within a week; the median within about **22 days** | Apply before feeling fully qualified |
| **Source matters** | **15–19 days** (SmartRecruiters, LinkedIn, company sites) vs. **28–37** on several platforms | Source adjusts urgency |
| **Some listing traits carry signal; keyword flags don't** | Disclosed pay, long descriptions and hiring platforms: **11–16 points** more still open after a week; urgency and remote wording: **no gap**. In the scope, EDA_SDA_2 finds these partly reflect crawl timing and collection (3.4, 3.9) | Model the traits; test richer NLP than keyword flags |
| **The best-paid field is the hardest to enter** | Computer and mathematical: **$66.11 an hour** median pay, **9%** junior, **77%** mid/senior | Target it, and hide what a career changer can't get |
| **How open listings are to career changers** | Entry level ranges from **3%** (management) to **70%** (food preparation); **22%** of computer and mathematical listings show any sign of openness | Its own ranking factor |

What separates a listing worth applying to today is mostly **the listing itself** (employer volume, source, pay disclosure), not the job title: those gaps (2.3–2.5×) are bigger than the gap between fields (1.7×), and all are known the moment a listing appears. **Update from EDA_SDA_2:** these are raw gaps; like-for-like the volume effect is about 1.4–1.5×, and much of the rest is crawl timing and listings with no description, which close in about 3 days.

### Decisions
- **Scope:** English listings in the United States, first seen April–September 2026; the 2,522 from the data's first two days (late March) are kept and flagged `partial_month`, left out of per-month counts only (hide, never remove). Listings with no language label are dropped, since they can't be confirmed; listings with no country are kept, flagged, since they mention remote work twice as often (10a); EDA_SDA_2 keeps those with a US-based company or remote wording.
- **Nothing capped:** every listing keeps its full follow-up (ghost listing vs. struggling employer is Part 2's question; EDA_SDA_2 3.1 hides likely ghosts and finds no sign that struggling employers welcome newcomers), and reposts become a feature.
- **Target: computer and mathematical**, split into role buckets in EDA_SDA_2. Manager-level listings are hidden only for applicants without management experience; text blockers (years asked, licenses) only after EDA_SDA_3 validates them.
- **This table keeps every group, seniority level and candidate factor,** so Part 2 can compare before narrowing.

### Limits
- **Six months of history,** so no seasonal advice. `first_seen` follows the crawler, so day-of-week patterns need `posted` (41% of listings).
- **We see listings appear and disappear, not hires.** Openness to career changers is what listings *ask for*, not who gets hired.
- **The text flags are keyword rules** with likely false positives ("urgent care") until EDA_SDA_3 validates them.
- **The six-month window misses older listings:** listings first seen before 2026-03-30 aren't in the data, so the earliest dates undercount what was open; the app's date picker starts at July 1 (8b).
- **The gaps here are descriptive, not significance tests.** Kaplan–Meier medians and shares show how big each gap is; whether it's statistically significant is tested in **EDA_SDA_4, section 5** (Q1d's chart shows one preview: a Gehan–Wilcoxon test, the survival version of the Wilcoxon test, since days open is censored): Mann–Whitney (Wilcoxon rank-sum) for skewed measures, log-rank for survival curves, with effect sizes and Benjamini–Hochberg correction.
- **High-volume posters include staffing agencies,** which can inflate company volume; EDA_SDA_2 (2b) flags them.

Detailed findings: section 9. Every decision with its evidence: section 10.

---

### Work plan

**Prioritization factors (candidates).** Each is a proxy measured from the listing data; the EDA shows which carry usable signal. For a career changer, the first question is *can I get it*; the rest answer *is it worth applying now*.

| Factor | Question for the career changer | Measured from |
|---|---|---|
| **Career-change accessibility** | Is it realistically open to a newcomer? Hard blockers hide it; softer signals raise it | `seniority`; description: years required, degree, "or equivalent experience", entry-level and career-changer wording |
| **Still open / apply-by** | Is it still worth applying, and by when? | `first_seen_at`, `last_seen_at` (100% filled) |
| **Freshness** | Am I early? | Days since `first_seen_at` |
| **Competition** | How many others am I up against? | Reposts; listings per company |
| **Pay** | Is it worth the move? | `salary_data` (about 43%), converted to an hourly rate |
| **Employer intent** | In a hurry or flexible? | Description and title: urgency, remote, length |
| **Source** | Where is it posted, and how often should I check? | `url`: platform, job board or company site |
| **Transferable fit** *(later)* | Do my old field's skills match? | Listing text vs. the career changer's profile, with O*NET skills; needs profiles (`Profiles.ipynb`) |

**Clarifying questions**
- How complete and clean is the data, and how far back does it go?
- How long do listings stay open, and what separates fast from slow ones (company volume, source, pay, reposts, text)?
- How open are listings to career changers: entry level, years asked, degrees?
- What time information is reliable: posting dates, crawler dates, months, weekdays?
- Which scope (language, country, months, segment) has enough data for depth?

| # | Question | What it decides |
|---|---|---|
| Q1 | Which factors should drive prioritization for a career changer? | The factors that vary enough to change the ranking, including career-change accessibility (Q1g) |
| Q2 | What time information can prioritization rely on? | Apply-by estimates, check cadence and seasonal context: which are supported |
| Q3 | Which segment do we go in depth on? | The target is computer and mathematical (decided 2026-10-09); Q3 shows how much of it a career changer can reach |

**Data rules:** aggregates only (counts, shares, quantiles). The one exception is the final-table preview in 11d, tagged `data-rows`; clear outputs before committing.

### Notebook roadmap

1. **Setup** · 2. **Load raw data** · 3. **Inspect structure** · 4. **Flatten** · 5. **Schema**: each one call into `src/pl_data.py`, on the first 200,000 documents
6. **Data cleaning:** types, invalid dates, duplicates and reposts, salary units, missing values, career-change flags
7. **Prioritization EDA:** Q1 factors (Q1g: career-change accessibility), Q2 time information, Q3 segment
8. **Other feasibility checks** · 9. **Findings** · 10. **Decisions, defended** · 11. **Final filters** · 12. **Summary statistics** · 13. **Conclusion**

## 1. Setup

Libraries and the data loader, `src/pl_data.py`: the S3 connection, cached raw documents, resumable flattening and the schema writer. Sections 2–5 each call one of its functions.

In [ ]:
# 1a Libraries and the data loader (src/pl_data.py), plus the full-file rows table (src/full_rows.py)
import gc
import json
import re
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT_ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "requirements.txt").exists())
sys.path.insert(0, str(PROJECT_ROOT))  # so `src` can be imported from notebooks/
from src.pl_data import (DATA_DIR, FLAT_DIR, JOBS_N, describe_structure, flatten_all, load_flat, load_raw, map_parts,
                         write_schemas)
from src.full_rows import (bit_counts, cat_contains, cat_map, has_label, load_rows, relabel, row_extras, save_final,
                           with_missing)

## 2. Load raw data

Both datasets are daily snapshots in S3, stored as compressed JSON lines (`.jsonl.zst`): **companies** (~252 MB compressed) are loaded in full; **job openings** (~19 GB) are too big for 8 GB of RAM, so we read the first 200,000 documents. Each is cached as Parquet in `data/`, so later runs take seconds instead of about 10 minutes. Once every table is flattened (section 4), this step skips the raw documents and S3 entirely.

**Sections 2–5 use these 200,000 documents** to show the raw structure, flatten it and write the schemas. **The analysis from section 6 on uses the full file:** `src/stream_full.py` reads all 13.4 million listings from S3 line by line and saves 6.1e's cleaned row for every English (or unlabeled) US (or no-country) listing, 13,017,464 rows (`python -m src.stream_full --rows`).

In [ ]:
# 2a Load the raw documents ({} when every table is already flattened)
raw = load_raw()

## 3. Inspect structure

Each line of a file is a JSON:API document with two keys: `data` (the records) and `included` (related objects the records point to, such as the company behind a job opening). Column names and Python types only.

In [ ]:
# 3a Column names and the Python type of each value in the first row
describe_structure(raw)

## 4. Flatten

Turns the nested documents into ordinary tables, one row per record, with nested fields as dotted columns (for example `attributes.first_seen_at`): the companies and job openings, and each file's related objects with duplicates removed. Each table is built in parts of 20,000 records saved to `data/flat/` as they finish, so after a crash a rerun skips finished parts, and a finished table is never rebuilt.

In [ ]:
# 4a Flatten all four tables (skipped when already done)
flatten_all(raw)

## 5. Schema

Writes a `df.info()`-style summary of every flattened table (column names, non-null counts, dtypes) into `docs/data.md`, only when something changed.

In [ ]:
# 5a Write the schemas into docs/data.md
write_schemas()

## 6. Data cleaning

Cleaning comes before any analysis, because every timing factor depends on valid dates. **Data-quality fixes happen here** (impossible dates, duplicates, placeholders, mixed units, inconsistent labels). **Scope decisions wait for section 10** and are applied together in section 11, since they're judgment calls that need the full data as evidence. Reposts are kept as a feature, not removed. Each step shows summary tables only and logs the rows it touched; 6.7 has the full cleaning report.

### 6.0 Chart style

One style for every chart in the notebook: a light surface, recessive grid, thin marks, and a fixed color order (blue, orange, aqua, yellow) so a color always means the same thing within a chart. Charts show aggregates only.

In [ ]:
# 6.0a Chart colors and style shared by every chart below
import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"  # categorical order: never reorder
INK, INK_2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#c9c8c2", "#e4e3df", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": INK_2, "ytick.color": INK_2, "text.color": INK,
    "font.size": 10.5, "lines.linewidth": 2, "legend.frameon": False, "figure.dpi": 110,
})

In [ ]:
# 6.0b Chart helpers
def bar_h(ax, s, title, xlabel, fmt="{:,.0f}", color=BLUE):
    """Horizontal bars, largest at the top, each value labelled at the bar end."""
    s = s.dropna().sort_values()
    ax.barh([str(i)[:42] for i in s.index], s.values, color=color, height=0.65)
    for y, v in enumerate(s.values):
        ax.text(v, y, " " + fmt.format(v), va="center", color=INK_2, fontsize=8.5)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.grid(axis="y", visible=False)
    ax.margins(x=0.18)

def finish():
    plt.tight_layout()
    plt.show()

*No output: this cell only sets the chart style.*

### 6.1 Cleaned rows and text cleaning

The stream (`src/stream_full.py --rows`) ran 6.1e on every listing in the file and saved the cleaned rows as typed Parquet, without text. This section loads them.

**Decisions, and why:**
- **Descriptions are cleaned** (HTML removed, entities decoded, whitespace collapsed), so word counts measure words, not markup.
- **Descriptions become features:** `has_desc`, `has_html`, `desc_words`, `mentions_remote`, `urgency`, and `desc_hash` (the repost fingerprint for 6.4). *Why:* memory; the full text stays in the stream's scope parts for NLP.
- **Career-change flags:** `years_required` (most years asked, lower end of each range; blank if none; above 20 ignored), `degree_required`, `equivalent_experience`, `entry_wording`, `changer_wording`. *Why:* what a listing asks of a newcomer is known the moment it appears. They're keyword rules, so EDA_SDA_3 checks them for false positives.
- **Without a description, text flags are unknown (blank), not False:** "we can't tell" isn't "it doesn't say so". `mentions_remote` is still True when the title says remote. Comparisons (Q1f, Q1g) set listings with a flag against listings *known* not to have it.
- **The `url` becomes its source:** a hiring platform, a job board, or the company's own site. Full URLs aren't needed.
- **13 million rows in 8 GB of RAM:** only light columns are held in memory (dates, numbers, flags, repeated text as categories). Titles, descriptions and list fields are reduced part by part to fingerprints, counts and flags (`src/full_rows.py`), and the text and list columns are added back from disk when the final table is saved (11c).

**Every field in the original job-openings table: kept or left out, and why** (34 flattened columns)

| Column | Description |
|:--|:--|
| `id` | Listing id. **Kept** (`id`): links the final table to the scope parts and later notebooks |
| `type` | Record type, the same for every listing. **Left out:** carries no information |
| `attributes.title` | Title as posted. **Kept** (`title`), plus `title_norm` (lowercased, spaces collapsed) to match reposts |
| `attributes.translated_title` | Title translated by PredictLeads; almost always empty. **Kept as a flag** (`was_translated`); the text isn't needed, since the final table is English only |
| `attributes.normalized_title` | PredictLeads' standardized title. **Kept** (`job_title`), falling back to the posted title where it's missing (`job_title_fallback`) |
| `attributes.description` | Full listing text. **Left out of this table** (too large for memory); reduced to `has_desc`, `has_html`, `desc_words`, the text flags and the repost fingerprint `desc_hash`. The scope listings keep the full text for EDA_SDA_3 |
| `attributes.url` | Link to the listing. **Reduced** to `source_type` and `platform`; full URLs aren't needed |
| `attributes.first_seen_at` | When the crawler first found the listing. **Kept** (`first_seen`): freshness, `days_open`, `month` |
| `attributes.last_seen_at` | When the crawler last saw it. **Kept** (`last_seen`): `live` and `days_open` |
| `attributes.last_processed_at` | When PredictLeads last processed the listing. **Kept** (`last_processed`) for a leakage audit only: it likely moves with `last_seen`, so it's future information until checked |
| `attributes.contract_types` | Full time, part time, remote and so on (a list). **Kept** (`contract_types`, labels unified) |
| `attributes.categories` | PredictLeads category labels (a list). **Kept** (`categories`) |
| `attributes.onet_data.code` | O*NET-SOC occupation code. **Kept** (`onet_code`): role buckets and job growth join on it |
| `attributes.onet_data.family` | O*NET occupation group. **Kept** (`occupation_group`): the scope cut |
| `attributes.onet_data.occupation_name` | O*NET occupation name. **Kept** (`occupation`) |
| `attributes.posted_at` | The employer's posting date, where given. **Kept** (`posted`); implausible dates blanked (6.3) |
| `attributes.recruiter_data.name` | Recruiter's name. **Left out:** personal data; only the yes/no flag `has_recruiter` is kept |
| `attributes.recruiter_data.title` | Recruiter's job title. **Left out:** personal data (folded into `has_recruiter`) |
| `attributes.recruiter_data.contact` | Recruiter's contact details. **Left out:** personal data (folded into `has_recruiter`) |
| `attributes.salary` | Pay as written in the listing (text). **Left out:** `salary_data` holds the same pay already parsed into numbers |
| `attributes.salary_data.salary_low` | Low end of the pay range, in the posted currency. **Kept** (`salary_low`) for the feature screening |
| `attributes.salary_data.salary_high` | High end of the pay range, in the posted currency. **Kept** (`salary_high`) |
| `attributes.salary_data.salary_currency` | Currency of the posted pay. **Kept** (`salary_currency`) |
| `attributes.salary_data.salary_low_usd` | Low end converted to USD. **Kept** (`salary_low_usd`) |
| `attributes.salary_data.salary_high_usd` | High end converted to USD. **Kept** (`salary_high_usd`) |
| `attributes.salary_data.salary_time_unit` | Hour, day, week, month or year. **Kept** (`salary_unit`): converts pay to `salary_hourly_usd` (6.5) |
| `attributes.seniority` | Seniority level (junior, mid_senior, manager and so on). **Kept** (`seniority`) |
| `attributes.status` | PredictLeads status: only "closed" or empty. **Kept** (`status`), but not used as a live flag: `live` comes from `last_seen` |
| `attributes.language` | Listing language. **Kept** (`language`): the English-only filter |
| `attributes.location` | Location as written (text). **Left out:** `location_data` holds the same location, structured |
| `attributes.location_data` | Structured locations (a list of places, each with its country and finer fields). **Reduced** to `country` (first location) and `n_locations`; EDA_SDA_2 reads the state from the scope listings (3.7d) |
| `attributes.tags` | PredictLeads tags (a list; about half empty). **Kept** (`tags`) |
| `relationships.company.data.id` | The hiring company's id. **Kept** (`company_id`): company volume, links to the companies table |
| `relationships.company.data.type` | Linked record type, the same for every listing. **Left out:** carries no information |

The file's related objects (`job_openings_included`: each company's domain, name and stock ticker) aren't in this table either; company facts come from the companies table through `company_id` (EDA_SDA_2).

In [ ]:
# 6.1a Paths and the columns we keep
CLEAN_DIR = DATA_DIR / "clean"
CLEAN_DIR.mkdir(exist_ok=True)

from src.listing_rows import COLS  # flattened column -> short name; shared with the full-file stream (src/stream_full.py)

In [ ]:
# 6.1b Patterns used to clean and flag description text. They live in src/listing_rows.py so this notebook and the
# full-file stream use one copy; EDA_SDA_3 validates them. Text: TAG_RE (HTML tags), URGENT_RE; career-change
# accessibility: YEARS_RE (with MAX_YEARS = 20, since "50 years of experience" is the company's history), ENTRY_RE,
# CHANGER_RE, DEGREE_RE, EQUIV_RE.
from src.listing_rows import (CHANGER_RE, DEGREE_RE, ENTRY_RE, EQUIV_RE, MAX_YEARS, TAG_RE, URGENT_RE, YEARS_RE,
                              years_required)

In [ ]:
# 6.1c Posting source from the url domain: hiring platforms (applicant tracking systems) and job boards, matched against
# the url's domain (ATS and BOARDS in src/listing_rows.py); everything else is "company site / other".
from src.listing_rows import ATS, BOARDS, source_of

In [ ]:
# 6.1d Text and location helpers: clean_text strips HTML and decodes entities; text_flag is True / False from the
# description and unknown (NA) without one; first_country takes the first of a listing's locations.
from src.listing_rows import clean_text, first_country, text_flag

In [ ]:
# 6.1e Turn one flattened part into lean rows: listing columns, text flags (unknown without a description), description
# length, the repost fingerprint (desc_hash) and the posting source. Shared with the full-file stream.
from src.listing_rows import lean_part

In [ ]:
# 6.1f Load the cleaned rows of the full file (src/stream_full.py --rows: 6.1e on every English US listing).
# Light columns only, so 13 million rows fit in 8 GB: repeated text as categories, flags as true / false / unknown.
# row_extras reads titles, ids and list fields part by part and keeps fingerprints, counts and bit flags instead.
LOAD_COLS = ["company_id", "first_seen", "last_seen", "posted", "status", "seniority", "language", "onet_code",
             "occupation_group", "occupation", "salary_low_usd", "salary_high_usd", "salary_unit", "has_desc", "has_html",
             "desc_words", "mentions_remote", "urgency", "years_required", "entry_wording", "changer_wording",
             "degree_required", "equivalent_experience", "n_locations", "country", "source_type", "platform"]
start = time.time()
stream = json.loads((FLAT_DIR / "stream_rows_progress.json").read_text())  # counts from the stream (no rows)
if not stream.get("finished"):
    raise RuntimeError("The rows stream hasn't finished: run `python -m src.stream_full --rows` first.")
jobs = load_rows(LOAD_COLS)
extras, labels = row_extras()  # same order as jobs: the index is each listing's row position in the rows table
jobs = pd.concat([jobs, extras], axis=1)
del extras
gc.collect()
print(f"loaded {len(jobs):,} cleaned rows × {jobs.shape[1]} columns in {time.time() - start:.0f}s, "
      f"{jobs.memory_usage(deep=True).sum() / 1e9:.1f} GB in memory")

companies = load_flat("companies", ["id"])

In [ ]:
# 6.1g Helpers for the cleaning report and summary tables
from IPython.display import display

report = []  # cleaning audit trail: one row per action, shown in 6.7
def log(step, action, n):
    report.append({"step": step, "action": action, "rows": int(n)})

def show(title, df):
    """Print a heading, then display a summary table. Aggregates only, never rows."""
    print(f"\n{title}")
    display(df)

def distinct(s):
    try:
        return s.nunique()
    except TypeError:  # list-valued columns (tags, categories, contract_types)
        return np.nan

In [ ]:
# 6.1h Summary tables
log("6.1", "rows in the cleaned rows table (every English or unlabeled, US or no-country listing)", len(jobs))

show(f"Cleaned rows: {len(jobs):,} rows × {jobs.shape[1]} columns in memory", pd.DataFrame({
    "dtype": jobs.dtypes.astype(str),
    "non-null %": (100 * jobs.notna().mean()).round(1),
    "distinct values": {c: distinct(jobs[c]) for c in jobs.columns},
}))
show("From the full file to the cleaned rows (the stream's filter, as decided in section 10)", pd.DataFrame({"listings": {
    "listings in the file": stream["records"],
    "English or unlabeled, and US or no country (kept)": stream["rows"],
    "other languages or countries (not kept)": stream["records"] - stream["rows"],
}}))

show("Text cleaning results (text flags: share of listings with a description)", pd.DataFrame({"share": {
    "has a description": jobs.has_desc.mean(),
    "description contained HTML (now stripped)": jobs.has_html.mean(),
    "mentions remote": jobs.mentions_remote.mean(),
    "urgency wording": jobs.urgency.mean(),
    "states years of experience required": jobs.years_required.notna().mean(),
    "entry-level or training wording": jobs.entry_wording.mean(),
    "career-changer wording": jobs.changer_wording.mean(),
    "degree required": jobs.degree_required.mean(),
    "or equivalent experience": jobs.equivalent_experience.mean(),
}}).round(3))

show("Years of experience required (listings that state it)",
     jobs.years_required.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_frame("years"))

show("Description length in words (postings with a description)",
     jobs.desc_words[jobs.has_desc].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(0).to_frame("words"))

show("Posting source (from the url domain)", pd.DataFrame({
    "postings": jobs.source_type.value_counts(),
    "share": jobs.source_type.value_counts(normalize=True).round(3),
}))

In [ ]:
# 6.1i Charts: description length and where listings are hosted
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1.4, 1]})
ax1.hist(jobs.desc_words[jobs.has_desc].clip(upper=2000), bins=50, color=BLUE)
ax1.axvline(400, color=INK_2, ls="--", lw=1)
ax1.text(420, ax1.get_ylim()[1] * 0.92, "~400 words: where small\nembedding models cut off", color=INK_2, fontsize=8.5, va="top")
ax1.set(title="Description length", xlabel="words per description (capped at 2,000)", ylabel="listings")
bar_h(ax2, 100 * jobs.source_type.value_counts(normalize=True), "Where listings are hosted", "% of listings", "{:.1f}%")
finish()

**What the data shows**
- **200,000 listings from 51,006 companies.** 82.3% have a description, only 0.3% had HTML, and the median is 607 words, so later embedding work needs chunking.
- **Up to 60.7% of listings live on company sites** (strictly "company site or other": any domain not on our platform and job-board lists), then hiring platforms (29.0%) and job boards (10.3%, mostly LinkedIn). Most listings can link straight to the employer's own page.
- **Among listings with a description, 13.1% mention remote work and 11.4% use urgency wording.** `status` and `country` have a single value (6.2, Q3).
- **Career-change flags:** 26.4% of listings state years of experience required (median 3; middle half 1–5; 10% ask 8 or more). Among listings with a description, 10.0% use entry-level or training wording, 6.6% accept equivalent experience, 3.7% require a degree, and only 0.4% use career-changer wording.

### 6.2 Types and standard categories

**Decisions, and why:**
- **Dates become UTC timestamps,** so they can be compared. Text that won't parse becomes missing, and is counted: guessing would invent data.
- **Category labels are trimmed and lowercased,** and `contract_types` spellings are unified ("full_time" = "full time"), so one value counts once.
- **`title_norm`** (lowercased, whitespace collapsed) is used to match reposts: built part by part into the repost fingerprint (6.1f) and saved with the final table (11c).
- **Dates were parsed in the stream** (text that wouldn't parse became missing there).

In [ ]:
# 6.2a Dates: parsed to UTC timestamps in the stream (text that wouldn't parse became missing there)
date_rows = {col: {"has a date": jobs[col].notna().sum(), "missing": jobs[col].isna().sum(),
                   "earliest": jobs[col].min(), "latest": jobs[col].max()} for col in ["first_seen", "last_seen", "posted"]}
show("Dates", pd.DataFrame(date_rows).T)

In [ ]:
# 6.2b Standardize category labels (on each column's distinct labels, not row by row)
cat_rows, label_tables = {}, {}
for col in ["status", "seniority", "language", "salary_unit"]:
    before = jobs[col].nunique()
    jobs[col] = relabel(jobs[col], lambda c: str(c).strip().lower())
    log("6.2", f"{col}: distinct labels merged by trimming and lowercasing", before - jobs[col].nunique())
    cat_rows[col] = {"labels before": before, "labels after": jobs[col].nunique(),
                     "merged": before - jobs[col].nunique(), "missing %": round(100 * jobs[col].isna().mean(), 1)}
    counts = with_missing(jobs[col]).value_counts()
    label_tables[col] = counts[counts > 0].head(8)
show("Category labels: trimmed and lowercased", pd.DataFrame(cat_rows).T)
for col, counts in label_tables.items():
    show(f"Top {col} labels after cleaning", pd.DataFrame({"postings": counts, "share": (counts / len(jobs)).round(3)}))

In [ ]:
# 6.2c Normalized titles and unified contract_types labels: both built part by part in 6.1f (row_extras).
# The normalized title (lowercased, whitespace collapsed) goes into the repost fingerprint (6.4) and is saved as title_norm
# (11c). contract_types spellings of the same label ("full_time" and "full time") are unified, kept as bit flags here.
before, after = labels["raw_contract_labels"], len(labels["contract_types"])
log("6.2", "contract_types: distinct labels merged (underscores, case, spacing)", before - after)
show("contract_types labels", pd.DataFrame({"labels": {"before": before, "after": after, "merged": before - after}}))
show("Titles", pd.DataFrame({"listings": {"with a title": jobs.has_title.sum(), "without a title": (~jobs.has_title).sum()}}))

**What the data shows**
- **Every date parsed.** `first_seen` spans exactly 2026-03-30 to 2026-09-30: a rolling six-month window.
- **`posted` has placeholders** (the earliest is 1969-12-30, blanked in 6.3), and its timestamps sit at midnight Central European time, so it's probably a local date (Q2c).
- **`status` is never "open"** (64.5% closed, 35.5% missing), so `last_seen` decides what's live (6.7). Seniority is 53% mid/senior and 30% junior; English is 92.4%.

### 6.3 Invalid dates

**Decisions, and why:**
- **Missing `first_seen` or `last_seen`, `last_seen` before `first_seen`, or dates after the snapshot: removed.** These are data errors, and such a listing can't be placed in time.
- **`posted` more than a day after `first_seen`, or before 2000: only `posted` is blanked.** The crawler can't find a listing before it exists, and 1969-12-30 is a placeholder; the listing itself stays.

In [ ]:
# 6.3a Remove impossible timelines
SNAPSHOT = pd.Timestamp("2026-09-30", tz="UTC")
END_OF_SNAPSHOT = SNAPSHOT + pd.Timedelta(days=1)
rows_before = len(jobs)

rules = {
    "missing first_seen or last_seen": jobs.first_seen.isna() | jobs.last_seen.isna(),
    "last_seen before first_seen": jobs.last_seen < jobs.first_seen,
    "dates after the snapshot": (jobs.first_seen > END_OF_SNAPSHOT) | (jobs.last_seen > END_OF_SNAPSHOT),
}
rule_rows, bad = {}, pd.Series(False, index=jobs.index)
for name, mask in rules.items():
    new = (mask & ~bad).sum()  # count each row once, under the first rule it fails
    log("6.3", f"removed, {name}", new)
    rule_rows[name] = {"rows": new, "share of rows": round(new / rows_before, 4), "action": "removed"}
    bad |= mask
jobs = jobs[~bad]
del rules
gc.collect()

In [ ]:
# 6.3b Blank implausible posted dates
bad_posted = (jobs.posted > jobs.first_seen + pd.Timedelta(days=1)) | (jobs.posted < pd.Timestamp("2000-01-01", tz="UTC"))
jobs.loc[bad_posted, "posted"] = pd.NaT
log("6.3", "posted blanked (after first_seen or before 2000)", bad_posted.sum())
rule_rows["posted after first_seen or before 2000"] = {
    "rows": bad_posted.sum(), "share of rows": round(bad_posted.sum() / rows_before, 4), "action": "posted blanked"}

In [ ]:
# 6.3c Summary tables
show("Date validity rules", pd.DataFrame(rule_rows).T)
show("Rows before and after", pd.DataFrame({"rows": {"before": rows_before, "removed": rows_before - len(jobs),
                                                     "after": len(jobs)}}))
days_open = ((jobs.last_seen - jobs.first_seen).dt.total_seconds() / 86400).astype("float32")
show("Days between first_seen and last_seen (after cleaning)",
     days_open.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_frame("days"))

In [ ]:
# 6.3d Chart: days between first seen and last seen
still_open = (SNAPSHOT - jobs.last_seen).dt.days <= 2
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.hist([days_open[~still_open], days_open[still_open]], bins=60, stacked=True, color=[BLUE, ORANGE],
        label=["closed by the snapshot", "still open at the snapshot"])
ax.set(title="Days between first seen and last seen", xlabel="days", ylabel="listings")
ax.legend()
finish()

**What the data shows**
- **No impossible timelines:** PredictLeads' timeline fields are clean.
- **5,454 `posted` values (2.7%) were blanked.**
- **Raw durations:** median 16.4 days; the maximum, 183.7 days, equals the six-month window. These treat still-open listings as ended; Q1b corrects that with survival curves.

### 6.4 Duplicates and reposts

**Decisions, and why:**
- **The same record `id` twice:** one copy kept (a data error).
- **Reposts** (same company, normalized title and description fingerprint, new `id`; matched on `repost_fp`, one 64-bit fingerprint of the three) **are kept and flagged** with `is_repost`; the earliest copy is the original. Features use **earlier copies only**, so they're known when a listing appears: `times_listed`, `days_since_first_listed`, `days_since_repost_last_listed`. Whether a listing *will* be reposted is future information and isn't used. Listings without a description can't be matched.
- **The EDA tables count each job once** (originals, `jo` in 6.7): a counting view, not a filter.

In [ ]:
# 6.4a Drop exact duplicate ids; flag reposts of the same job
# Both use 64-bit fingerprints built in 6.1f (id_fp; repost_fp = company, normalized title and description fingerprint):
# one number per listing instead of long text keys. Two different keys sharing a fingerprint is about a 1-in-200,000
# chance across all 13 million listings.
rows_before = len(jobs)
dup_id = jobs.id_fp.duplicated()
log("6.4", "removed, duplicate id", dup_id.sum())
jobs = jobs[~dup_id].drop(columns="id_fp")

jobs = jobs.sort_values("first_seen")
key = ["repost_fp"]
has_key = jobs.repost_fp.notna()
jobs["is_repost"] = has_key & jobs.duplicated(key)
log("6.4", "flagged as repost (kept, excluded from the EDA tables)", jobs.is_repost.sum())
log("6.4", "postings without a description, can't be checked for reposts", (~has_key).sum())

# Repost features: built only from earlier copies of the same job, so they're known when a listing appears
same_job = jobs[has_key].groupby(key)
jobs["times_listed"] = np.float32(np.nan)  # unknown without a description
jobs.loc[has_key, "times_listed"] = (same_job.cumcount() + 1).astype("float32")  # jobs are sorted by first_seen
jobs["days_since_first_listed"] = np.float32(np.nan)  # since the job's original listing
jobs.loc[has_key, "days_since_first_listed"] = (
    jobs.loc[has_key, "first_seen"] - same_job.first_seen.transform("min")).dt.total_seconds().div(86400).astype("float32")
jobs["days_since_repost_last_listed"] = np.float32(np.nan)  # since the job's previous listing; 0 for the first listing
jobs.loc[has_key, "days_since_repost_last_listed"] = (
    jobs.loc[has_key, "first_seen"] - same_job.first_seen.shift()).dt.total_seconds().fillna(0).div(86400).astype("float32")
del same_job
gc.collect()

In [ ]:
# 6.4b Summary tables
show("Duplicates and reposts", pd.DataFrame({"rows": {
    "before": rows_before,
    "exact duplicate ids (removed)": dup_id.sum(),
    "after": len(jobs),
    "reposts (flagged, kept)": jobs.is_repost.sum(),
    "originals": (~jobs.is_repost).sum(),
    "no description, can't be checked": (~has_key).sum(),
}}))
group_sizes = jobs[has_key].groupby(key).size()
show("How many times the same job is listed (original + reposts)",
     group_sizes.value_counts().sort_index().head(10).rename_axis("listings per job").to_frame("jobs"))
show("Repost features: times this job has been listed so far",
     jobs.times_listed.fillna(0).astype(int).clip(upper=5).value_counts().sort_index()
     .rename(index={0: "unknown (no description)", 5: "5+"}).rename_axis("times listed").to_frame("listings"))
show("Repost rate by source type", jobs.groupby("source_type", observed=True).is_repost.mean().round(3).to_frame("repost rate"))

In [ ]:
# 6.4c Charts: repost rate and listings per job
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.2))
bar_h(ax1, 100 * jobs.groupby("source_type", observed=True).is_repost.mean(), "Repost rate by source type", "% of listings that are reposts", "{:.1f}%")
sizes = group_sizes.clip(upper=10).value_counts().sort_index()
ax2.bar([str(int(k)) if k < 10 else "10+" for k in sizes.index], sizes.values, color=BLUE, width=0.65)
ax2.set_yscale("log")
ax2.set(title="Listings per job (log scale)", xlabel="times the same job is listed", ylabel="jobs")
ax2.grid(axis="x", visible=False)
finish()

**What the data shows**
- **No duplicate ids, and 4,199 reposts (2.1%):** a lower bound, since 35,490 listings without a description can't be checked.
- **Most jobs are listed once,** but a long tail is relisted 10 or more times (1,268 listings are a job's 5th listing or later).
- **Job boards repost far more** (8.4%) than company sites (1.8%) or hiring platforms (0.5%).

### 6.5 Salary to one hourly rate

**Decisions, and why:**
- **The midpoint of the low and high values, converted to an hourly rate** (a year is 2,080 hours, 260 workdays, 52 weeks or 12 months), so hourly and salaried pay compare. Many career changers come from hourly jobs, so hourly reads naturally. Hourly listings keep their posted rate; converting salaried pay assumes full-time work.
- **Reversed ranges** (low above high) are swapped, and **clearly wrong units** are re-read: "$25 a year" is hourly pay, "$85,000 an hour" is yearly pay. Each is flagged `pay_suspect`.
- **A missing or unknown unit, or a result outside $4.81–$480.77 an hour** ($10k–$1M a year): only the salary is blanked, and counted.

In [ ]:
# 6.5a Salary time units
PER_YEAR = {"hour": 2080, "day": 260, "week": 52, "month": 12, "year": 1}

def unit_factor(u):
    if pd.isna(u):
        return np.nan
    return next((f for k, f in PER_YEAR.items() if k in u), np.nan)

mid = jobs[["salary_low_usd", "salary_high_usd"]].mean(axis=1)
factor = cat_map(jobs.salary_unit, unit_factor)
unit_counts = with_missing(jobs.loc[mid.notna(), "salary_unit"]).value_counts()
unit_counts = unit_counts[unit_counts > 0]
show("Salary units (postings with a salary)", pd.DataFrame({
    "postings": unit_counts,
    "× per year": unit_counts.index.map(lambda u: unit_factor(u) if u != "(missing)" else np.nan),
}).fillna({"× per year": "not recognized"}))

In [ ]:
# 6.5b Fix reversed ranges and wrong units, convert to one hourly rate, blank outliers
HOURS_PER_YEAR = PER_YEAR["hour"]
reversed_range = jobs.salary_low_usd > jobs.salary_high_usd
jobs.loc[reversed_range, ["salary_low_usd", "salary_high_usd"]] = (
    jobs.loc[reversed_range, ["salary_high_usd", "salary_low_usd"]].to_numpy())
log("6.5", "salary range reversed (low above high): swapped", reversed_range.sum())

mid = jobs[["salary_low_usd", "salary_high_usd"]].mean(axis=1)
yearly_but_hourly = cat_contains(jobs.salary_unit, "year") & mid.between(7, 200)              # "$25 a year" is hourly pay
hourly_but_yearly = cat_contains(jobs.salary_unit, "hour") & mid.between(10_000, 1_000_000)   # "$85,000 an hour" is yearly pay
factor = cat_map(jobs.salary_unit, unit_factor).mask(yearly_but_hourly, PER_YEAR["hour"]).mask(hourly_but_yearly, PER_YEAR["year"])
log("6.5", "labelled yearly but clearly hourly ($7-$200): re-read as hourly", yearly_but_hourly.sum())
log("6.5", "labelled hourly but clearly yearly ($10k-$1M): re-read as yearly", hourly_but_yearly.sum())
log("6.5", "salary present but unit missing or not recognized", (mid.notna() & factor.isna()).sum())

hourly = mid * factor / HOURS_PER_YEAR
outlier = ~hourly.between(10_000 / HOURS_PER_YEAR, 1_000_000 / HOURS_PER_YEAR) & hourly.notna()
jobs["salary_hourly_usd"] = hourly.mask(outlier).astype("float32")
jobs["pay_suspect"] = reversed_range | yearly_but_hourly | hourly_but_yearly
log("6.5", "hourly pay outside $4.81-$480.77 ($10k-$1M a year), blanked", outlier.sum())

In [ ]:
# 6.5c Summary tables
show("Salary cleaning", pd.DataFrame({"postings": {
    "salary present": mid.notna().sum(),
    "range reversed (swapped)": reversed_range.sum(),
    "labelled yearly, re-read as hourly": yearly_but_hourly.sum(),
    "labelled hourly, re-read as yearly": hourly_but_yearly.sum(),
    "unit missing or not recognized": (mid.notna() & factor.isna()).sum(),
    "converted to hourly": hourly.notna().sum(),
    "outside $4.81–$480.77 an hour (blanked)": outlier.sum(),
    "usable hourly pay": jobs.salary_hourly_usd.notna().sum(),
}}))
q = [0.1, 0.25, 0.5, 0.75, 0.9]
show("Hourly pay (USD) quantiles, before and after removing outliers", pd.DataFrame({
    "before": hourly.quantile(q), "after": jobs.salary_hourly_usd.quantile(q)}).round(2))

In [ ]:
# 6.5d Charts: salary units and hourly pay
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 1.4]})
bar_h(ax1, unit_counts, "Salary time units", "listings with a salary")
pay = jobs.salary_hourly_usd.dropna().clip(upper=150)
ax2.hist(pay, bins=60, color=BLUE)
ax2.axvline(pay.median(), color=INK_2, ls="--", lw=1)
ax2.text(pay.median(), ax2.get_ylim()[1] * 0.95, f"  median ${pay.median():.2f}/hr", color=INK_2, fontsize=8.5, va="top")
ax2.set(title="Hourly pay (midpoint, full-time equivalent, capped at $150)", xlabel="USD per hour", ylabel="listings")
finish()

**What the data shows**
- **86,154 listings (43.1%) disclose pay,** all with a recognized unit: hourly is most common (48,215), then yearly (35,120).
- **Only 14 outliers** fell outside $4.81–$480.77 an hour ($10k–$1M a year), and no reversed ranges or wrong units turned up.
- **The median is about $29.11 an hour** (middle half $20.00–$50.00), with two humps: hourly and salaried jobs.

### 6.6 Missing values

The share of postings missing each field we plan to use. It shows what each analysis can rely on. For example, the crawler-vs-employer check in 7.1 only uses postings that have `posted`.

In [ ]:
# 6.6 Missing-value summary for the fields the analysis uses
fields = ["first_seen", "last_seen", "posted", "status", "seniority", "language",
          "occupation_group", "occupation", "salary_hourly_usd", "country", "platform"]
missing = pd.DataFrame({"missing %": 100 * jobs[fields].isna().mean(), "available rows": jobs[fields].notna().sum()})
missing.loc["title"] = [100 * (~jobs.has_title).mean(), jobs.has_title.sum()]
missing.loc["description"] = [100 * (~jobs.has_desc).mean(), jobs.has_desc.sum()]
show("Missing values (after cleaning)", missing.round(1).sort_values("missing %", ascending=False))

fig, ax = plt.subplots(figsize=(8, 4))
bar_h(ax, missing["missing %"], "Missing values by field", "% of listings missing", "{:.1f}%")
finish()

**What the data shows**
- **The timeline fields, title, seniority and occupation group are essentially complete,** so factors built on them cover every listing.
- **`posted` (58.9% missing) and salary (56.9%) are the weak fields;** `platform` is missing for 60.7% by design (company sites have none).
- **`country` is missing for 8.4%;** every listing that has one is in the United States.

### 6.7 Cleaning report and derived timing columns

**Decisions, and why:**
- **`live`: last seen within 2 days of the snapshot,** since the crawler doesn't revisit every listing daily. A live listing's lifespan is *censored*, which Q1b handles.
- **`days_open`** (first seen to last seen) and **`month`** (month first seen, the time horizon).
- **`jo`: originals only, for the EDA tables;** every cleaned listing, reposts flagged, is saved to disk and starts the final table in section 11 (memory: one copy in RAM at a time).
- **Career-change flags:** `entry_seniority`, `low_experience_ask` (2 years or fewer) and `career_open` (either of those, or entry-level/training or career-changer wording): a deliberately broad first pass for Q1g and Q3. "Or equivalent experience" is left out, since it usually means experience in the same field.

In [ ]:
# 6.7a Derived timing columns and the analysis table (originals only)
LIVE_DAYS = 2
jobs["live"] = (SNAPSHOT - jobs.last_seen).dt.days <= LIVE_DAYS
jobs["days_open"] = ((jobs.last_seen - jobs.first_seen).dt.total_seconds() / 86400).astype("float32")
jobs["month"] = jobs.first_seen.dt.tz_localize(None).dt.to_period("M")
jobs["occupation_group"] = with_missing(jobs.occupation_group)

# Career-change accessibility flags (see the decisions above)
ENTRY_LEVELS = {"junior"}  # checked in Q3b: junior is the only entry-level label (others: mid_senior, manager, head, director, vice_president, president, c_level, partner, founder, not_set)
jobs["entry_seniority"] = jobs.seniority.isin(ENTRY_LEVELS)
jobs["low_experience_ask"] = jobs.years_required <= 2  # False where no years are stated
jobs["career_open"] = jobs.entry_seniority | jobs.low_experience_ask | jobs.entry_wording | jobs.changer_wording

# Memory: every cleaned listing (reposts flagged) goes to disk for section 11; jobs keeps only the columns the
# repost and file-order checks below need, and jo (originals) is read back from disk.
CLEANED_PATH = CLEAN_DIR / "cleaned_rows_full.parquet"
jobs.to_parquet(CLEANED_PATH)
JOBS_COLS = ["company_id", "first_seen", "occupation_group", "source_type", "is_repost", "repost_fp", "language",
             "country", "has_desc", "mentions_remote", "contract_bits"]
jobs = jobs[JOBS_COLS]
gc.collect()
jo = pd.read_parquet(CLEANED_PATH, filters=[("is_repost", "==", False)])
log("6.7", "postings in the analysis table (originals only)", len(jo))

In [ ]:
# 6.7b Cleaning report and analysis table at a glance
show("Cleaning report (every action, in order)", pd.DataFrame(report))
start_rows = report[0]["rows"]
show("From raw to analysis table", pd.DataFrame({
    "rows": {"cleaned rows (6.1)": start_rows, "after date checks (6.3) and duplicates (6.4)": len(jobs),
             "originals, used for the EDA (6.7)": len(jo)},
}).assign(**{"share of start": lambda d: (d.rows / start_rows).round(3)}))
show("Analysis table at a glance", pd.DataFrame({"value": {
    "postings": len(jo),
    "companies": jo.company_id.nunique(),
    "occupation groups": jo.occupation_group.nunique(),
    "still open at snapshot": jo.live.sum(),
    "first_seen from": jo.first_seen.min().date(),
    "first_seen to": jo.first_seen.max().date(),
}}))

**What the data shows**
- **Cleaning removed almost nothing:** no rows were dropped, and 4,199 reposts were set aside, leaving **195,801 original listings** for the EDA.
- **42,399 listings (21.7%) are still open at the snapshot.** That's the pool a prioritized list would rank today.
- **The main fixes were to fields, not rows:** placeholder `posted` dates, salary units, and `contract_types` labels.

## 7. Prioritization EDA

Q1 measures every candidate factor (ending with career-change accessibility in Q1g), Q2 checks which time information the data supports, and Q3 compares segments. All use the cleaned originals (`jo`); repost rates use `jobs`.

### 7.0 Helpers

In [ ]:
# 7.0a Shared constants and the largest occupation groups
COMPANY = "company_id"
TOP = 15
TOP_GROUPS = 10
top_groups = pd.Index(jo.occupation_group[jo.occupation_group != "(missing)"].value_counts().index[:TOP_GROUPS].astype(str))
WEEKS = max((jo.first_seen.max() - jo.first_seen.min()).days / 7, 1)  # weeks covered by the sample

In [ ]:
# 7.0b Small helpers
def as_list(v):
    """Nested list fields come back as lists or numpy arrays; normalize to a list."""
    if isinstance(v, (list, tuple, np.ndarray)):
        return list(v)
    return [] if v is None or (isinstance(v, float) and np.isnan(v)) else [v]

def pct(x):
    return f"{100 * x:.1f}%"

In [ ]:
# 7.0c Kaplan–Meier survival helpers
def km_curve(days, finished):
    """Kaplan–Meier curve: share of postings still open after t days, indexed by t.
    Live postings count as censored, so the open ones don't bias the window short."""
    d = pd.DataFrame({"t": days, "e": finished}).dropna()
    by_t = d.groupby("t")["e"].agg(["size", "sum"])
    at_risk = len(d) - by_t["size"].cumsum().shift(fill_value=0)
    return (1 - by_t["sum"] / at_risk).cumprod()

def surv_at(curve, x):
    return curve[curve.index <= x].iloc[-1] if (curve.index <= x).any() else 1.0

def km_survival(days, finished, at):
    curve = km_curve(days, finished)
    return [surv_at(curve, x) for x in at]

def km_median(days, finished):
    """Days until half of postings have closed; NaN if more than half are still open."""
    curve = km_curve(days, finished)
    return curve[curve <= 0.5].index.min()

### 7.1 Q1: Which factors should drive prioritization?

A factor only helps ranking if it **varies**, so Q1a shows each factor overall, across the largest occupation groups, and its spread (max ÷ min). Q1b–Q1g look closer.

| Measure | Prioritization factor | What it tells an applicant | Caveat |
|---|---|---|---|
| Median days open; closed within 7 days | Still open / apply-by | How long, and how urgent, the window is | Disappearing ≠ filled |
| New listings per week | Freshness, source | How often to check | Depends on crawl frequency |
| Still open at snapshot | Still open | How many listings are rankable now | One day only |
| Repost rate | Competition | Jobs that didn't fill | Could also mean a bad job |
| Salary disclosed, median salary | Pay | Whether it's worth applying | Under half disclose pay |
| Urgency wording, mentions remote | Employer intent | In a hurry or flexible | Keyword flags |
| From a hiring platform | Source | Where listings are hosted | From the url domain |
| Top 10 companies' share | Competition, source | Whether a short watch list covers most listings | Sample may over-represent some companies |
| Entry-level seniority; asks 2 years or fewer; open to career changers | Career-change accessibility | Whether a newcomer can get in | Keyword rules; validated in EDA_SDA_3 |

In [ ]:
# 7.1 Q1a: define the candidate prioritization factors
def targets(g, g_all):
    """Candidate prioritization factors for one group. g = originals, g_all = originals + reposts."""
    curve = km_curve(g.days_open, ~g.live)
    return pd.Series({
        "median days open (Kaplan–Meier)": curve[curve <= 0.5].index.min(),
        "closed within 7 days": 1 - surv_at(curve, 7),
        "new postings per week": len(g) / WEEKS,
        "still open at snapshot": g.live.sum(),
        "repost rate": g_all.is_repost.mean(),
        "salary disclosed": g.salary_hourly_usd.notna().mean(),
        "median hourly pay (USD)": g.salary_hourly_usd.median(),
        "urgency wording": g.urgency.mean(),
        "mentions remote": g.mentions_remote.mean(),
        "from a hiring platform": (g.source_type == "hiring platform").mean(),
        "top 10 companies' share": g[COMPANY].value_counts().head(10).sum() / len(g),
        "entry-level seniority": g.entry_seniority.mean(),
        "asks 2 years or fewer": g.low_experience_ask.mean(),
        "open to career changers (any sign)": g.career_open.mean(),
    })

In [ ]:
# Q1a: compute the factors overall and per occupation group
by_group = pd.DataFrame({f: targets(jo[jo.occupation_group == f], jobs[jobs.occupation_group == f]) for f in top_groups}).T
overview = pd.DataFrame({
    "overall": targets(jo, jobs),
    "group min": by_group.min(),
    "group median": by_group.median(),
    "group max": by_group.max(),
})
overview["spread (max / min)"] = overview["group max"] / overview["group min"].replace(0, np.nan)
show("Q1a. Candidate prioritization factors: overall, and the range across the largest occupation groups", overview.round(2))
show("Q1a. Candidate prioritization factors by occupation group", by_group.round(2))

In [ ]:
# Q1a chart: how much each factor differs between occupation groups
fig, ax = plt.subplots(figsize=(8, 4))
bar_h(ax, overview["spread (max / min)"], "How much each factor differs between the largest occupation groups",
      "spread: highest group ÷ lowest group", "{:.1f}×")
ax.axvline(1, color=INK_2, lw=1)
finish()

**What the data shows**
- **Biggest gaps between groups:** remote 15.5× (2% food preparation to 35% computer and mathematical, among listings with a description), reposts 10.3× (0.9% to 9.0%), pay 3.6× ($18.46 to $66.11 an hour).
- **Moderate:** new listings per week 3.1×, urgency 2.2×, days open 1.7× (15.7 to 26.1). Salary disclosure and hiring-platform share barely differ between groups, but still separate listings within one (Q1c, Q1f).
- **So remote, reposts and pay change a lot by field,** and the model must be tuned per segment.
- **Career-change accessibility varies most of all:** entry-level seniority 20.7× (3% management to 70% food preparation), open to career changers 4.6× (17% to 79%), asks 2 years or fewer 2.3× (7% computer and mathematical to 15%).

#### Q1b. Still open / apply-by

The share of listings still open N days after first being seen, using Kaplan–Meier survival curves. These treat still-open listings as unfinished rather than ignoring them, which would make windows look shorter than they are. This is the basis for an **apply-by estimate** on each listing.

In [ ]:
# 7.1 Q1b: application window by occupation group
DAYS = [3, 7, 14, 30, 60]
window = {f: km_survival(g.days_open, ~g.live, DAYS) for f, g in jo[jo.occupation_group.isin(top_groups)].groupby("occupation_group", observed=True)}
window["(all postings)"] = km_survival(jo.days_open, ~jo.live, DAYS)
show("Q1b. Share of listings still open N days after first seen (Kaplan–Meier)",
     pd.DataFrame(window, index=[f"open after {d}d" for d in DAYS]).T.loc[list(top_groups) + ["(all postings)"]].round(2))

In [ ]:
# Q1b chart: survival curves for the fastest and slowest occupation groups
grid = np.arange(0, 121)
def on_grid(curve):
    return curve.reindex(curve.index.union(grid)).ffill().reindex(grid).fillna(1.0)

medians = {f: km_median(g.days_open, ~g.live) for f, g in jo[jo.occupation_group.isin(top_groups)].groupby("occupation_group", observed=True)}
fast, slow = min(medians, key=medians.get), max(medians, key=medians.get)
fig, ax = plt.subplots(figsize=(9.5, 4.2))
for f, g in jo[jo.occupation_group.isin(top_groups)].groupby("occupation_group", observed=True):
    if f not in (fast, slow):
        ax.step(grid, on_grid(km_curve(g.days_open, ~g.live)), where="post", color=MUTED, lw=1)
ax.plot([], [], color=MUTED, lw=1, label="other large occupation groups")
for f, color, tag in ((fast, ORANGE, "fastest"), (slow, BLUE, "slowest")):
    g = jo[jo.occupation_group == f]
    ax.step(grid, on_grid(km_curve(g.days_open, ~g.live)), where="post", color=color, label=f"{f} ({tag})")
ax.step(grid, on_grid(km_curve(jo.days_open, ~jo.live)), where="post", color=INK, lw=1.5, ls="--", label="all listings")
ax.set(title="Share of listings still open, by days since first seen", xlabel="days since first seen",
       ylabel="share still open", xlim=(0, 120), ylim=(0, 1.02))
ax.legend(fontsize=8.5)
finish()

**What the data shows**
- **The window is short:** 28% of listings are gone within 7 days, 59% within 30 and 78% within 60; the median within about 22 days.
- **It differs by field:** transportation and computer and mathematical lose 34% in the first week; food preparation 22%.
- **For prioritization:** a week's delay costs a fifth to a third of the options, so apply-by estimates must be specific to the occupation group.

#### Q1c. Source

Which hiring platforms, job boards or company sites host listings, and whether arrival rate and window differ by source. The crawl covers company sites and platforms, so job boards are under-represented.

In [ ]:
# 7.1 Q1c: sources: share of postings, arrival rate, window
show("Q1c. Source type", pd.DataFrame({"listings": jo.source_type.value_counts(),
                                       "share": jo.source_type.value_counts(normalize=True).round(3)}))
source = jo.platform.astype(object).fillna(jo.source_type.astype(object)).astype("category")  # platform, else source type
by_src = jo[["days_open", "live"]].groupby(source, observed=True).apply(lambda g: pd.Series({
    "postings": len(g),
    "share": len(g) / len(jo),
    "new per week": len(g) / WEEKS,
    "median days open (Kaplan–Meier)": km_median(g.days_open, ~g.live),
    "open after 7d": km_survival(g.days_open, ~g.live, [7])[0],
}))
show("Q1c. Platforms and other sources: volume, arrival rate, window", by_src.sort_values("postings", ascending=False).head(TOP).round(2))

In [ ]:
# Q1c: source mix within the largest occupation groups
mix = pd.crosstab(jo.occupation_group, source, normalize="index").loc[top_groups]
show("Q1c. Source mix within the largest occupation groups (% of each occupation group's listings)",
     (100 * mix[mix.sum().sort_values(ascending=False).index[:8]]).round(1))

In [ ]:
# Q1c chart: median days open by source
fig, ax = plt.subplots(figsize=(8.5, 4.6))
top_src = by_src.sort_values("postings", ascending=False).head(TOP)
bar_h(ax, top_src["median days open (Kaplan–Meier)"], "Median days a listing stays open, by source (15 largest)", "days (Kaplan–Meier median)", "{:.0f} d")
finish()

**What the data shows**
- **Median days open runs from 15 (SmartRecruiters), 17 (LinkedIn) and 19 (company sites) to 33–37** (Paylocity, Greenhouse, BambooHR).
- **Company sites close fastest:** 66% still open after a week vs. 82–89% on most platforms. The mix is similar in every group (company sites host 55–69%), so source separates listings, not fields.
- **Caveat:** platforms may show closures to the crawler differently, and platform choice tracks employer size (Q1d), so source is tested alongside company volume.

#### Q1d. Competition: company hiring volume

Whether a few companies post most listings (a short watch list for an applicant), and whether listings from high-volume companies close faster or slower.

In [ ]:
# 7.1 Q1d: company concentration by occupation group
def concentration(g):
    per_co = g[COMPANY].value_counts()
    per_co = per_co[per_co > 0]  # company_id is a category: drop companies with no listings in this group
    return pd.Series({
        "companies": per_co.size,
        "postings per company (median)": per_co.median(),
        "top 1% of companies' share": per_co.head(max(1, per_co.size // 100)).sum() / len(g),
        "top 10% of companies' share": per_co.head(max(1, per_co.size // 10)).sum() / len(g),
        "companies with 5+ postings": (per_co >= 5).sum(),
    })
show("Q1d. Company concentration by occupation group",
     jo.loc[jo.occupation_group.isin(top_groups), [COMPANY, "occupation_group"]]
     .groupby("occupation_group", observed=True).apply(concentration, include_groups=False).loc[top_groups].round(2))

In [ ]:
# Q1d: do high-volume companies' listings close faster or slower?
co_postings = jo[COMPANY].value_counts()
def size_band(company_ids):
    return pd.cut(cat_map(company_ids, co_postings).fillna(1), [0, 1, 4, 19, np.inf],
                  labels=["1 posting", "2-4", "5-19", "20+"])
by_band = jo[["days_open", "live"]].groupby(size_band(jo[COMPANY]), observed=True).apply(lambda g: pd.Series({
    "share of postings": len(g) / len(jo),
    "median days open (Kaplan–Meier)": km_median(g.days_open, ~g.live),
}))
by_band["repost rate"] = jobs.groupby(size_band(jobs[COMPANY]), observed=True).is_repost.mean()
show("Q1d. Do high-volume companies' listings close faster or slower?", by_band.round(2))

In [ ]:
# Q1d check: volume counted from earlier listings only, the way the model will see it
# The table above counts each company's listings over all six months, including ones posted later (descriptive only).
# Here each listing's band counts only the company's listings first seen before it (same-day listings count together).
BANDS = ["1 posting", "2-4", "5-19", "20+"]
earlier = jo.groupby(COMPANY, observed=True).first_seen.rank(method="min") - 1
asof_band = pd.cut(earlier + 1, [0, 1, 4, 19, np.inf], labels=BANDS)
since_may = jo.first_seen >= pd.Timestamp("2026-05-01", tz=jo.first_seen.dt.tz)  # a month or more of history behind each

def band_medians(mask):
    return jo.loc[mask, ["days_open", "live"]].groupby(asof_band[mask], observed=True).apply(lambda g: km_median(g.days_open, ~g.live))

show("Q1d check: median days open (Kaplan–Meier) by company volume, whole six months vs. earlier listings only", pd.DataFrame({
    "whole six months (descriptive)": by_band["median days open (Kaplan–Meier)"],
    "earlier listings only": band_medians(pd.Series(True, index=jo.index)),
    "earlier only, first seen May or later": band_medians(since_may),
    "share of listings (earlier only)": asof_band.value_counts(normalize=True),
}).reindex(BANDS).round(2))

In [ ]:
# Q1d chart: survival curves by company volume (earlier listings only), with a Gehan–Wilcoxon test of 20+ vs. 1 posting
# Each curve is the Kaplan–Meier share of listings still open after each day; a median is where a curve crosses 50%.
# Days open is censored (still-open listings haven't finished), so the Wilcoxon test used here is the survival version:
# a log-rank test weighted toward early differences (Gehan–Breslow–Wilcoxon). Formal tests are in EDA_SDA_4; this is a preview.
# The test runs on a random 500,000 listings per group (the full groups would take too long); the curves use every listing.
from lifelines.statistics import logrank_test
TEST_N = 500_000
fig, ax = plt.subplots(figsize=(9, 4.2))
for band, color in zip(BANDS, [MUTED, AQUA, ORANGE, BLUE]):
    g = jo.loc[asof_band == band, ["days_open", "live"]]
    curve = km_curve(g.days_open, ~g.live)
    ax.step(curve.index, 100 * curve.values, where="post", color=color, lw=1.8,
            label=f"{band} (median {km_median(g.days_open, ~g.live):.0f} days, {len(g):,} listings)")
ax.axhline(50, color=INK_2, ls="--", lw=0.8)
ax.text(1, 51.5, "median: half still open", color=INK_2, fontsize=8)
a, b = (jo.loc[asof_band == band, ["days_open", "live"]] for band in ("20+", "1 posting"))
a, b = (g.sample(min(len(g), TEST_N), random_state=0) for g in (a, b))
test = logrank_test(a.days_open, b.days_open, event_observed_A=~a.live, event_observed_B=~b.live, weightings="wilcoxon")
ax.set(title=f"Share still open by company volume (earlier listings only)\nGehan–Wilcoxon test, 20+ vs. 1 posting "
             f"(random {TEST_N:,} each): statistic {test.test_statistic:,.0f}, "
             f"p {'< 0.001' if test.p_value < 0.001 else f'= {test.p_value:.3f}'}",
       xlabel="days since first seen", ylabel="% still open", xlim=(0, 120), ylim=(0, 100))
ax.legend(fontsize=8, frameon=False)
finish()

In [ ]:
# Q1d charts: days open and share of listings, by company hiring volume
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.2))
order = ["1 posting", "2-4", "5-19", "20+"]
by_band = by_band.reindex(order)
ax1.bar(order, by_band.loc[order, "median days open (Kaplan–Meier)"], color=BLUE, width=0.6)
for x, v in enumerate(by_band.loc[order, "median days open (Kaplan–Meier)"]):
    ax1.text(x, v, f"{v:.0f} d", ha="center", va="bottom", color=INK_2, fontsize=8.5)
ax1.set(title="Median days open, by company hiring volume", xlabel="listings the company has in the file", ylabel="days")
ax1.grid(axis="x", visible=False)
ax2.bar(order, 100 * by_band.loc[order, "share of postings"], color=BLUE, width=0.6)
ax2.set(title="Share of all listings", xlabel="listings the company has in the file", ylabel="% of listings")
ax2.grid(axis="x", visible=False)
finish()

**What the data shows**
- **The strongest single signal, and it holds:** listings from companies with 20+ openings stay open a median 14 days vs. 32 for one-off listings when volume counts all six months. Counting only each company's **earlier** listings (the way the model will see it), it's **13 vs. 30**, and 13 vs. 33 among listings first seen from May, with a month or more of history. Fewer listings land in the top band that way (26% vs. 42%), since a company's early listings count as low volume.
- **High-volume employers post 42% of all listings;** in every group the top 10% of companies post 39–60% (healthcare practitioners is the most concentrated).
- **Watching about 300–1,100 companies per group** (those with 5+ listings) covers a large share of openings. Reposts are more common there too (3% vs. 1%).

#### Q1e. Competition: reposts

A repost is the same job (company, title and description) listed again under a new id. It suggests the first listing didn't fill, which could mean less competition, or a hard-to-fill or unattractive role. Here we check how common reposts are and how long the gap is between original and repost.

In [ ]:
# 7.1 Q1e: repost rates and gaps
show("Q1e. Repost rate by occupation group", jobs[jobs.occupation_group.isin(top_groups)].groupby("occupation_group", observed=True).agg(
    listings=("is_repost", "size"), reposts=("is_repost", "sum"), repost_rate=("is_repost", "mean")
).loc[top_groups].round(3))

In [ ]:
# Q1e: days from the original listing to each repost
first = jobs[~jobs.is_repost & jobs.repost_fp.notna()].groupby("repost_fp").first_seen.min()
reposts = jobs.loc[jobs.is_repost, ["repost_fp", "first_seen"]].join(first.rename("original_seen"), on="repost_fp")
gap = (reposts.first_seen - reposts.original_seen).dt.total_seconds() / 86400
show("Q1e. Days from the original to each repost",
     gap.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_frame("days"))

In [ ]:
# Q1e charts: repost rate and repost gap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1.3, 1]})
rates = jobs[jobs.occupation_group.isin(top_groups)].groupby("occupation_group", observed=True).is_repost.mean()
bar_h(ax1, 100 * rates, "Repost rate by occupation group", "% of listings that are reposts", "{:.1f}%")
ax2.hist(gap.clip(upper=120), bins=40, color=BLUE)
ax2.set(title="Days from original to repost", xlabel="days (capped at 120)", ylabel="reposts")
finish()

**What the data shows**
- **Reposting is concentrated in hourly, high-turnover fields:** food preparation (9.0%), transportation (4.9%) and sales (3.1%). Most other occupation groups sit near 1%.
- **Reposts come back quickly:** a median of 16 days after the original, and a quarter within 6 days.
- **For prioritization:** a repost is a useful competition signal in those three fields. The employer didn't fill the role the first time. Elsewhere it's too rare to matter.

#### Q1f. Employer intent: text signals (the NLP link)

A first look at whether simple text features go with listings staying open longer or shorter. Each row compares listings with and without the feature. These are keyword flags, not models, and they misfire ("urgent care" counts as urgency, "remote patient monitoring" as remote), so EDA_SDA_3 checks a sample of matches. If any of them show a gap, richer text features (TF-IDF, topics, embeddings) are worth modeling as prioritization inputs.

In [ ]:
# 7.1 Q1f: do text features relate to how long postings stay open?
flags = {
    "salary disclosed": jo.salary_hourly_usd.notna(),
    "mentions remote": jo.mentions_remote,
    "urgency wording": jo.urgency,
    "long description (top 25%)": jo.desc_words > jo.desc_words.quantile(0.75),
    "on a hiring platform": jo.source_type == "hiring platform",
}
rows = {}
for name, mask in flags.items():
    known = mask.notna().to_numpy()                  # text flags are unknown without a description
    has = mask.fillna(False).astype(bool).to_numpy()
    with_f = km_survival(jo.days_open[has], ~jo.live[has], [7, 30])
    without = km_survival(jo.days_open[known & ~has], ~jo.live[known & ~has], [7, 30])
    rows[name] = {"share of postings": has[known].mean(), "open 7d (with)": with_f[0], "open 7d (without)": without[0],
                  "open 30d (with)": with_f[1], "open 30d (without)": without[1]}
show("Q1f. Text features vs. application window (all occupation groups)", pd.DataFrame(rows).T.round(2))

In [ ]:
# Q1f chart: share still open after 7 days, with vs. without each feature
feat = pd.DataFrame(rows).T.sort_values("open 7d (with)")
fig, ax = plt.subplots(figsize=(8.5, 3.4))
y = np.arange(len(feat))
ax.hlines(y, feat["open 7d (without)"], feat["open 7d (with)"], color=MUTED, lw=2)
ax.scatter(feat["open 7d (without)"], y, color=INK_2, s=60, zorder=3, label="without the feature")
ax.scatter(feat["open 7d (with)"], y, color=BLUE, s=60, zorder=3, label="with the feature")
ax.set_yticks(y, feat.index)
ax.set(title="Share of listings still open after 7 days", xlabel="share still open")
ax.grid(axis="y", visible=False)
ax.legend(fontsize=8.5, loc="lower right")
finish()

**What the data shows**
- **Disclosed pay, long descriptions and hiring platforms go with listings staying open longer** (share still open after a week, with vs. without): hiring platform 83% vs. 67%, long description 81% vs. 69%, salary disclosed 78% vs. 67%.
- **Urgency and remote wording show no gap** once listings are compared only with listings *known* not to use them: urgency 79% vs. 79%, remote 76% vs. 79%. Earlier versions showed gaps (79% vs. 71%, 76% vs. 71%) because listings without a description, which close faster, were counted as "no".
- **So keyword flags alone carry little timing signal;** description length and richer text features (EDA_SDA_3) are still worth testing. One feature at a time, and they overlap with company size and source, so a multivariate model (Cox regression) must separate them.

#### Q1g. Career-change accessibility

How open each occupation group is to someone new to the field, and whether open listings behave differently. Text measures use listings with a description.

| Measure | What it tells a career changer | Caveat |
|---|---|---|
| Entry-level seniority | Whether there are roles a newcomer can start in | PredictLeads' label; its rules aren't documented |
| States years required; median years asked; asks 2 years or fewer | How much field experience listings expect | Lower end of each range |
| Entry-level or training wording; career-changer wording | Whether the employer trains or invites newcomers | Keyword rules |
| Degree required; "or equivalent experience" | Whether a missing degree rules the applicant out | One listing can say both |
| Open to career changers (any sign) | The first-pass summary flag | Broad on purpose |
| Median pay | Whether the field is worth the move | Under half disclose pay |

In [ ]:
# 7.1 Q1g: career-change accessibility, by occupation group
def access(g):
    """Career-change accessibility for one group; text measures use listings with a description."""
    d = g[g.has_desc]
    stated = d.years_required.dropna()
    return pd.Series({
        "listings": len(g),
        "entry-level seniority": g.entry_seniority.mean(),
        "states years required": d.years_required.notna().mean(),
        "median years required (where stated)": stated.median(),
        "asks 2 years or fewer (where stated)": (stated <= 2).mean(),
        "entry-level or training wording": d.entry_wording.mean(),
        "career-changer wording": d.changer_wording.mean(),
        "degree required": d.degree_required.mean(),
        "or equivalent experience": d.equivalent_experience.mean(),
        "open to career changers (any sign)": g.career_open.mean(),
        "median hourly pay (USD)": g.salary_hourly_usd.median(),
        "median hourly pay (USD), open listings only": g.loc[g.career_open.fillna(False), "salary_hourly_usd"].median(),
    })

ACCESS_COLS = ["has_desc", "entry_seniority", "years_required", "entry_wording", "changer_wording", "degree_required",
               "equivalent_experience", "career_open", "salary_hourly_usd"]  # only what access() reads (memory)
acc = pd.DataFrame({f: access(jo.loc[jo.occupation_group == f, ACCESS_COLS]) for f in top_groups}).T
acc.loc["all listings"] = access(jo[ACCESS_COLS])
show("Q1g. Career-change accessibility by occupation group (largest groups)", acc.round(2))

In [ ]:
# Q1g: do listings open to career changers stay open longer or shorter?
cc_flags = {
    "entry-level seniority": jo.entry_seniority,
    "asks 2 years or fewer": (jo.years_required <= 2).where(jo.years_required.notna()).astype("boolean"),  # only listings that state years
    "asks 5 years or more": (jo.years_required >= 5).where(jo.years_required.notna()).astype("boolean"),
    "entry-level or training wording": jo.entry_wording,
    "degree required": jo.degree_required,
    "open to career changers (any sign)": jo.career_open,
}
cc_rows = {}
for name, mask in cc_flags.items():
    known = mask.notna().to_numpy()                  # compare with vs. known-without, never vs. unknown
    has = mask.fillna(False).astype(bool).to_numpy()
    with_f = km_survival(jo.days_open[has], ~jo.live[has], [7, 30])
    without = km_survival(jo.days_open[known & ~has], ~jo.live[known & ~has], [7, 30])
    cc_rows[name] = {"share of listings": has[known].mean(), "open 7d (with)": with_f[0], "open 7d (without)": without[0],
                     "open 30d (with)": with_f[1], "open 30d (without)": without[1]}
show("Q1g. Career-change flags vs. application window (all occupation groups)", pd.DataFrame(cc_rows).T.round(2))

In [ ]:
# Q1g chart: how open each occupation group is to career changers, against pay
pts = acc.drop(index="all listings")
x, y = 100 * pts["open to career changers (any sign)"], pts["median hourly pay (USD)"]
fig, ax = plt.subplots(figsize=(9.5, 5))
ax.scatter(x, y, s=400 * pts["listings"] / pts["listings"].max(), color=BLUE, alpha=0.85, edgecolor=SURFACE)
for name in pts.index:
    ax.annotate(str(name)[:32], (x[name], y[name]), xytext=(6, 4), textcoords="offset points", fontsize=8, color=INK_2)
ax.axvline(100 * acc.loc["all listings", "open to career changers (any sign)"], color=MUTED, ls="--", lw=1)
ax.axhline(acc.loc["all listings", "median hourly pay (USD)"], color=MUTED, ls="--", lw=1)
ax.set(title="Open to career changers vs. pay, largest occupation groups (dashed: all listings)",
       xlabel="% of listings with any sign of being open to career changers", ylabel="median hourly pay ($)")
finish()

**What the data shows**
- **Fields differ widely.** Entry-level seniority runs from 3% (management) and 9% (computer and mathematical) to 70% (food preparation). Among listings that state years, 17% of computer and mathematical listings ask for 2 or fewer (median 5 years), vs. 74–80% in food preparation, healthcare support and healthcare practitioners.
- **Degrees are rarely required (4%),** and computer and mathematical listings accept equivalent experience most often (14%).
- **Listings open to career changers close a little faster:** 73% still open after a week vs. 80%. Computer and mathematical is low on openness (22% show any sign) but its open listings still pay a median $39.99 an hour, vs. $21.15 for open listings overall.
- **Caveat:** keyword flags, one at a time. EDA_SDA_3 validates the rules and EDA_SDA_4 separates the effects.

### 7.2 Q2: What time information can prioritization rely on?

| Time scale | Use in prioritization | Needs |
|---|---|---|
| Days since a listing appeared | Apply-by estimates and freshness | Closed listings with both dates |
| Week / day of week | How often to check for new listings | Dates that reflect employers, not the crawler |
| Season / month | When to search | About two years of history |

Checks: how far back `first_seen` goes; whether the file is sorted (which would bias the first-200,000 sample); and whether `posted` and `first_seen` weekdays differ (crawler vs. employer).

In [ ]:
# 7.2 Q2a: how much history does the snapshot hold?
show("Q2a. How much history does the snapshot hold?", pd.DataFrame({"value": {
    "first_seen from": jo.first_seen.min().date(),
    "first_seen to": jo.first_seen.max().date(),
    "distinct months": jo.month.nunique(),
    "complete years": jo.month.nunique() // 12,
    "share already closed by the snapshot (high = keeps history)": round((~jo.live).mean(), 3),
}}))
show("Q2a. Listings by first_seen year", jo.first_seen.dt.year.value_counts().sort_index()
     .rename_axis("year").to_frame("listings"))
show("Q2a. Listings by first_seen month (last 24)", jo.month.value_counts().sort_index().tail(24)
     .rename_axis("month").to_frame("listings"))

In [ ]:
# Q2b: is the file sorted by date or company?
in_file_order = jobs[[COMPANY, "first_seen"]].sort_index()  # index = row position in the file
order_corr = pd.Series(np.arange(len(in_file_order))).corr(
    in_file_order.first_seen.rank().reset_index(drop=True), method="spearman")
same_company = in_file_order[COMPANY] == in_file_order[COMPANY].shift()
show("Q2b. Is the file sorted?", pd.DataFrame({
    "value": {"rank correlation of file order with first_seen": round(order_corr, 2),
              "rows whose company matches the previous row": round(same_company.mean(), 3)},
    "how to read": {"rank correlation of file order with first_seen": "near 0 = not sorted by date",
                    "rows whose company matches the previous row": "high = grouped by company"},
}))

In [ ]:
# Q2c: employer or crawler? posted vs. first_seen
both = jo.posted.notna()
lag = (jo.first_seen - jo.posted)[both].dt.total_seconds() / 86400
show(f"Q2c. Days from posted to first_seen (listings with posted: {pct(both.mean())})",
     lag.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_frame("days"))
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday = pd.DataFrame({
    "posted (UTC)": jo.posted[both].dt.day_name().value_counts(normalize=True),
    # posted timestamps sit at 22:00/23:00 UTC = midnight Central European time, so read them as CET dates
    "posted (CET date)": jo.posted[both].dt.tz_convert("Europe/Berlin").dt.day_name().value_counts(normalize=True),
    "first_seen, same listings": jo.first_seen[both].dt.day_name().value_counts(normalize=True),
    "first_seen, all listings": jo.first_seen.dt.day_name().value_counts(normalize=True),
}).reindex(days)
show("Q2c. % of listings by weekday (if the columns disagree, first_seen reflects the crawler)",
     (100 * weekday).round(1))

In [ ]:
# Q2d: weekly arrivals
weekly = jo.first_seen.dt.tz_localize(None).dt.to_period("W").value_counts().sort_index()
show("Q2d. Weekly arrivals: enough weeks for a check cadence?", pd.DataFrame({"value": {
    "weeks with listings": weekly.size,
    "median new listings per week": weekly.median(),
    "complete years of monthly data (seasonal decomposition needs 2+)": jo.month.nunique() // 12,
}}))

In [ ]:
# Q2 chart: new listings per week
fig, ax = plt.subplots(figsize=(9.5, 3.4))
wk = weekly.copy()
partial = [wk.index[0], wk.index[-1]]  # first and last weeks are cut off by the window
ax.plot([p.start_time for p in wk.index], wk.values, color=BLUE, marker="o", ms=4)
ax.scatter([p.start_time for p in partial], wk[partial].values, color=MUTED, s=40, zorder=3, label="partial week")
ax.set(title="New listings per week (first seen)", ylabel="listings")
ax.legend(fontsize=8.5)
finish()

In [ ]:
# Q2 charts: weekday shares and the posted-to-first-seen lag
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.6), gridspec_kw={"width_ratios": [1.5, 1]})
x = np.arange(len(days))
for k, (col, color) in enumerate([("posted (CET date)", BLUE), ("first_seen, all listings", ORANGE)]):
    ax1.bar(x + (k - 0.5) * 0.38, 100 * weekday[col], width=0.36, color=color, label=col)
ax1.set_xticks(x, [d[:3] for d in days])
ax1.set(title="Share of listings by weekday", ylabel="% of listings")
ax1.grid(axis="x", visible=False)
ax1.legend(fontsize=8.5)
ax2.hist(lag.clip(-1, 14), bins=30, color=BLUE)
ax2.set(title="Days from posted to first seen", xlabel="days (capped at 14)", ylabel="listings")
finish()

In [ ]:
# Q2 chart: hour of day of first_seen: the crawler's schedule, not employers'
hours = jo.first_seen.dt.hour.value_counts(normalize=True).reindex(range(24), fill_value=0)
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(hours.index, 100 * hours.values, color=MUTED, width=0.7)
ax.set(title="Hour of day listings were first seen (UTC): when the crawler ran", xlabel="hour (UTC)", ylabel="% of listings")
ax.grid(axis="x", visible=False)
finish()

**What the data shows**
- **Only six months of history,** so no seasonal advice. Weekly volume is steady at about 7,200 new listings (a dip in June), and the file isn't sorted by date or company, so the sample shows no ordering bias.
- **`first_seen` is a good freshness clock** (found a median 1.5 days after posting; 90% within 8 days), **but its weekdays and hours follow the crawler** (Friday peak). Day-of-week patterns must use `posted` (Central European dates, 41% of listings); hour of day isn't supported.
- **For prioritization:** posting age is fully supported; a day-of-week factor uses `posted`; seasonal context is out of reach.

### 7.3 Q3: Which segment do we go in depth on?

**Decided 2026-10-09: computer and mathematical (US),** split in EDA_SDA_2 into role buckets defined the same way as the career-switch profiles. Q3 still compares every large group, to show how much of the target a career changer can reach and what the alternatives were. Criteria: openness to newcomers (`entry_share`, `career_open_share`), pay, listings and companies (`top_company_share`), measurable timing (`closed_share`, `posted` coverage), variation in the Q1 factors, text length (Q3e), and monthly variation (`monthly_cv`, a lead only, since sampling affects it). The source backgrounds (the team's own) aren't in the listing data; EDA_SDA_2 pairs them with targets.

In [ ]:
# 7.3 Q3a: occupation groups ranked by size
def group_stats(g):
    monthly = g.month.value_counts()
    return pd.Series({
        "postings": len(g),
        "companies": g[COMPANY].nunique(),
        "top_company_share": g[COMPANY].value_counts(normalize=True).iloc[0],
        "closed_share": (~g.live).mean(),
        "median_days_open_closed": g.loc[~g.live, "days_open"].median(),
        "posted_share": g.posted.notna().mean(),
        "salary_share": g.salary_hourly_usd.notna().mean(),
        "median_salary": g.salary_hourly_usd.median(),
        "entry_share": g.entry_seniority.mean(),
        "career_open_share": g.career_open.mean(),
        "months": monthly.size,
        "monthly_cv": monthly.std() / monthly.mean(),
    })

grp = jo[[COMPANY, "occupation_group", "month", "live", "days_open", "posted", "salary_hourly_usd", "entry_seniority",
          "career_open"]].groupby("occupation_group", observed=True).apply(group_stats, include_groups=False).sort_values("postings", ascending=False)
show("Q3a. Occupation groups ranked by size", grp.head(20).round(2))

In [ ]:
# Q3b–Q3d: seniority mix and countries (the stream kept only US or no-country listings: section 10 decided that from
# the first 200,000 documents, which had every country)
show("Q3b. Seniority mix of the largest occupation groups",
     pd.crosstab(jo.occupation_group, with_missing(jo.seniority)).loc[grp.index[:TOP]])

country_counts = with_missing(jo.country).value_counts()
show("Q3c. Countries (first listed location)",
     country_counts[country_counts > 0].head(TOP).rename_axis("country").to_frame("listings"))

top_countries = [c for c in jo.country.value_counts().index[:6] if (jo.country == c).any()]
show("Q3d. Listings by occupation group × country (largest occupation groups and countries)",
     pd.crosstab(jo.occupation_group, jo.country).loc[grp.index[:TOP], top_countries])

In [ ]:
# Q3 chart: size vs. closing speed
top15 = grp.drop(index="(missing)", errors="ignore").head(TOP)
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(top15.postings, top15.median_days_open_closed, s=60 + 540 * top15.companies / top15.companies.max(), color=BLUE, alpha=0.75,
           edgecolor=SURFACE, linewidth=2)
for name, r in top15.iterrows():
    ax.annotate(name[:30], (r.postings, r.median_days_open_closed), xytext=(6, 3), textcoords="offset points",
                fontsize=8, color=INK_2)
ax.set(title="Occupation groups: size vs. how fast closed listings closed (bubble size = companies)",
       xlabel="listings in the sample", ylabel="median days open (closed listings)")
finish()

In [ ]:
# Q3 chart: seniority mix
levels = {"junior": "junior", "mid_senior": "mid/senior", "manager": "manager"}
mix = pd.crosstab(jo.occupation_group, jo.seniority.astype(object).map(levels).fillna("director and above / other"), normalize="index")
mix = mix.reindex(index=grp.index[:10].drop("(missing)", errors="ignore"),
                  columns=["junior", "mid/senior", "manager", "director and above / other"]).fillna(0)
fig, ax = plt.subplots(figsize=(10, 4.2))
left = np.zeros(len(mix))
for col, color in zip(mix.columns, [BLUE, ORANGE, AQUA, YELLOW]):
    ax.barh([n[:40] for n in mix.index], 100 * mix[col], left=left, color=color, height=0.65,
            edgecolor=SURFACE, linewidth=2, label=col)
    left += 100 * mix[col].values
ax.invert_yaxis()
ax.set(title="Seniority mix of the largest occupation groups", xlabel="% of the occupation group's listings")
ax.grid(axis="y", visible=False)
ax.legend(ncol=4, fontsize=8.5, loc="lower center", bbox_to_anchor=(0.5, -0.32))
finish()

In [ ]:
# Q3e: words per description, by occupation group and by country
with_text = jo.loc[jo.has_desc, ["occupation_group", "desc_words"]].assign(country=with_missing(jo.country[jo.has_desc]))

def words_summary(g):
    w = g.desc_words
    return pd.Series({
        "listings with a description": len(g),
        "median words": w.median(),
        "mean words": w.mean(),
        "25th percentile": w.quantile(0.25),
        "75th percentile": w.quantile(0.75),
        "share over 400 words": (w > 400).mean(),
    })

words_by_group = (with_text.groupby("occupation_group", observed=True).apply(words_summary, include_groups=False)
                  .sort_values("median words", ascending=False))
show("Q3e. Words per description by occupation group", words_by_group.round(2))
show("Q3e. Words per description by country",
     with_text.groupby("country", observed=True).apply(words_summary, include_groups=False).round(2))

In [ ]:
# Q3e chart: median words per description by occupation group
fig, ax = plt.subplots(figsize=(8.5, 6.5))
bar_h(ax, words_by_group["median words"].drop("(missing)", errors="ignore"),
      "Median words per description, by occupation group", "words (listings with a description)")
finish()

**What the data shows**
- **Country isn't a choice:** every listing with a country is in the US. **Every large group has enough data:** the top ten have 8,000–25,500 listings from 3,000–12,000 companies, and no company holds more than 2%.
- **Computer and mathematical** (15,688 listings, 7,152 companies) has the highest pay ($66.11 an hour), the most remote listings (35% of those with a description), among the fastest closing (34% gone in a week) and the longest descriptions (median 749 words), **but only 9% junior and 77% mid/senior.** Alternatives: healthcare practitioners (largest and most concentrated, but many roles need a license; general knowledge, not measured) and sales or food preparation (junior-heavy, lower pay).
- **Decided:** computer and mathematical, hiding the listings a career changer can't get rather than switching fields. EDA_SDA_2 checks that enough remains.
- **Most open to career changers:** building and grounds cleaning (85% any sign, 77% entry level), food preparation (79%), transportation (78%) and personal care (72%), all with median pay under $22 an hour. Computer and mathematical: 22% open, 9% entry level, $66.11 an hour.

## 8. Other feasibility checks

Fields the project may use later: language (for an English-only corpus), status values, contract types, what `tags` and `categories` hold, and how many job openings link to a company in the companies table.

In [ ]:
# 8. Remaining feasibility checks: structure of other fields, company match
def counts(s, n=TOP):
    vc = s.value_counts()
    vc = vc[vc > 0].head(n)
    return pd.DataFrame({"listings": vc, "share": (vc / len(jo)).round(3)})

def label_counts(bits, names, n=TOP):
    vc = bit_counts(bits, names).head(n)
    return pd.DataFrame({"listings": vc, "share": (vc / len(jo)).round(3)})

show("Language", counts(with_missing(jo.language), 10))
show("Status values", counts(with_missing(jo.status)))
contract = label_counts(jo.contract_bits, labels["contract_types"])
contract.loc["(none)"] = [(jo.contract_bits == 0).sum(), round((jo.contract_bits == 0).mean(), 3)]
contract["listings"] = contract.listings.astype(int)
show("contract_types (a listing can have several)", contract)

show("tags and categories (structure only; distinct values counted over every cleaned listing)", pd.DataFrame({
    "avg items per listing": {"tags": round(jo.n_tags.mean(), 1), "categories": round(jo.n_categories.mean(), 1)},
    "empty share": {"tags": round((jo.n_tags == 0).mean(), 3), "categories": round((jo.n_categories == 0).mean(), 3)},
    "distinct values": {"tags": labels["distinct_tags"], "categories": len(labels["categories"])},
}))
show("Top categories", label_counts(jo.category_bits, labels["categories"]))

show("Company match rate", pd.DataFrame({"value": {
    "job openings whose company is in the companies table": round(jo[COMPANY].isin(companies["id"]).mean(), 3)}}))

In [ ]:
# 8 chart: contract types
fig, ax = plt.subplots(figsize=(8, 4))
ct = bit_counts(jo.contract_bits, labels["contract_types"])
ct.loc["(none)"] = (jo.contract_bits == 0).sum()
ct = ct.sort_values(ascending=False).head(12)
bar_h(ax, 100 * ct / len(jo), "Contract types (after cleaning; a listing can have several)", "% of listings", "{:.1f}%")
finish()

In [ ]:
# 8b Date picker check: listings live on the 1st of each month (counts only)
# Live on a date = first seen on or before it and last seen on or after it. The data starts on 2026-03-30, so listings
# that opened earlier are missing (left truncation): early dates should look thin and young.
rows = {}
for d in pd.date_range("2026-04-01", "2026-09-01", freq="MS", tz=jo.first_seen.dt.tz):
    age = (d - jo.first_seen[(jo.first_seen <= d) & (jo.last_seen >= d)]).dt.days  # listings live on that date
    rows[d.strftime("%Y-%m-%d")] = {"live listings": len(age), "median days since first seen": age.median(),
                                    "share first seen in the past 30 days": (age <= 30).mean()}
show("8b. Listings live on the 1st of each month", pd.DataFrame(rows).T.round(2))

**What the data shows**
- **English covers 92.3%** of listings, and `status` holds only "closed" or nothing.
- **`contract_types` needed unifying** (6.2); its onsite, remote and hybrid labels give a second remote signal.
- **`tags` are too sparse to use as skills** (53% empty, 7,196 values), while **`categories`** is a clean 27-value taxonomy (99% filled). Every job opening's company is in the companies table.
- **The date picker is fair from July (8b):** live listings jump from 2,459 on April 1 and 23,884 on May 1 to 32,000–40,000 from June on. Their age mix settles from July, when about half were first seen in the past 30 days (June: 66%; April: all of them). Earlier dates are skewed toward young listings because the data starts on 2026-03-30.

## 9. Findings: where to narrow down

**Q1, which factors drive prioritization**

| Factor | Evidence | Verdict |
|---|---|---|
| Competition: company hiring volume | 14 vs. 32 median days open over the whole six months, 13 vs. 30 counting earlier listings only (20+ openings vs. one-off); these companies post 42% of listings | **Strongest factor:** holds with earlier listings only (13 vs. 30 days, Q1d check); model it first |
| Source | 15 (SmartRecruiters), 17 (LinkedIn), 19 (company sites) to 37 (BambooHR) median days; 66% vs. 82–89% open after a week | **Strong:** model it with company volume, since the two overlap |
| Still open / apply-by | 28% gone in 7 days, median about 22 days; 22–34% in the first week by group | **Strong:** the basis for apply-by estimates; group-specific |
| Employer intent (text) | Long descriptions go with longer windows (81% vs. 69% open at 7 days); urgency and remote wording show no gap | **Mixed:** test description length and richer NLP; keyword flags alone carry little |
| Pay | Disclosed for 43%; disclosure goes with longer windows (78% vs. 67%); medians $18.46–$66.11 an hour by group | **Useful:** as a value factor, and disclosure as a timing signal |
| Competition: reposts | 2.1% overall; 3–9% in food preparation, transportation and sales | **Segment-dependent** |
| Freshness | `first_seen` lags `posted` by a median 1.5 days | **Reliable** for every listing |
| Career-change accessibility | Entry level, years required, degree, training and career-changer wording (entry level 3–70% by field; 22% of computer and mathematical listings show any sign of openness). Only 9% of computer and mathematical listings are junior | **Core:** hard blockers hide a listing; softer signals raise it. Validated in EDA_SDA_3 before anything is hidden |
| Transferable fit | Needs a career changer's profile | Not testable here; uses `Profiles.ipynb` and O*NET skills later |

**Source and the app:** up to 60.7% of listings are on a company site (the share includes any domain not on our platform lists), so the app can link most ranked listings straight to the employer's page; EDA_SDA_2 (3.2) checks the domains.

**Q2, time information:** posting age is supported for every listing; day of week only through `posted` (41%, Central European dates); seasonal context isn't supported with six months of history.

**Q3, segment:** computer and mathematical in the US (decided 2026-10-09). Only 9% of its listings are junior, so EDA_SDA_2 hides the listings a career changer can't get, splits the rest into role buckets, and checks that enough remains.

## 10. Final analysis: defending our decisions

Every decision about the final table, defended with the evidence from sections 6–9. Nothing is filtered until section 11 applies them all at once. *Decided* is applied in section 11; *Applied in EDA_SDA_2* is decided here and applied there; *Kept for now* and *Open* await later checks.

| Decision | Options | Recommendation | Choice | Status |
|:--|:--|:--|:--|:--|
| **Language** | All languages; English only | English only (92.4% of listings) | English only | Decided |
| **Listings with no language label** (4.6%) | Keep; drop | Drop: we can't confirm they're English | Drop | Decided. Check 10a: 99.8% of these have no description, so this drop also removes about a quarter of the listings without descriptions |
| **Country** | United States is the only country present | United States | United States | Decided; listings without a country are handled in the next row |
| **Listings with no country** (8.4%) | Keep; drop; keep and place them with the company's location | Keep, flagged `country_missing`, and resolve in EDA_SDA_2 | Keep, flagged | **Applied in EDA_SDA_2** (decided 2026-10-09): kept if the company is US-based (companies table) or the listing mentions remote work or uses the remote contract type, flagged `country_inferred`; the rest are dropped. *Why:* check 10a shows they mention remote twice as often (28.3% vs. 12.6%), and remote roles matter to career changers (35% of computer and mathematical listings) |
| **Occupation groups** | One **main** group plus **related** groups (Q3) | Main: computer and mathematical (decided 2026-10-09). Related: the team's source-background groups, kept for role discovery and comparison | Computer and mathematical, plus the source-background groups | **Applied at the start of EDA_SDA_2 (1d, 2026-10-10):** every other group is cut, leaving only a per-occupation summary of pay and timing (1c) for comparisons and the compass. Inside the scope, role bucket (from `onet_code`) is a dimension, not a filter; only buckets too thin on both the listings and the profiles side are dropped |
| **Seniority** | All levels; hide manager level for everyone; hide it per applicant; entry level only | Keep all levels; hide manager-level listings per applicant | Keep all, labelled | **Decided (2026-10-09):** manager-level listings are hidden only for applicants without management or senior experience, since prior experience can qualify a career changer. Entry level only would drop mid-level listings that ask for little experience. Text blockers (years asked, licenses) hide nothing until EDA_SDA_3 validates them |
| **Time horizon** | Weekly; monthly | Monthly, April–September; keep the 2-day partial March, flagged `partial_month` | Monthly | Decided (2026-10-10: late March kept and flagged instead of dropped; hide, never remove) |
| **Time scales** | Month; week; day of week; hour of day | Month, week and day of week | Month, week, day of week | Decided. Day of week uses `posted` (41%, Central European dates), since `first_seen` days follow the crawler. Hour of day isn't supported |
| **Follow-up window** | Drop listings open more than 50 days; cap at 50 days; keep the full record | Keep the full record | Keep the full record | Decided: a long-open listing could be a ghost listing (rank lower) **or** an employer still struggling to hire (rank higher). Age alone can't tell them apart; EDA_SDA_2 separates the two |
| **Prioritization factors (Q1)** | See below | | Keep all candidates for now | **Kept for now:** non-text questions are checked in EDA_SDA_2, text questions in EDA_SDA_3 |
| **Additional data cleaning** | | | Applied in EDA_SDA_2 | **Decided:** six fixes in EDA_SDA_2 section 2b (staffing agencies, salary sanity, date consistency, internships and temporary roles, title normalization, multi-location listings) and description cleanup in EDA_SDA_3; fix or flag, never drop |

### Prioritization factors (Q1)

| Factor | Choice | Reason | Status |
|:--|:--|:--|:--|
| Still open / apply-by | The model's **target**, not an input | It's what the ranking predicts from the other factors | Decided |
| Company hiring volume | Keep, counted from listings posted **before** each listing | Strongest signal (14 vs. 32 median days open). High volume could mean **more competition or more openings to win**, so its direction is still open | Kept; interpretation open |
| Source / platform | Keep for now: `source_type` and `platform`; keep job-board rows | Whether source matters beyond company volume, whether platform gaps reflect crawler detection, and whether job-board listings duplicate company-site ones are checked in EDA_SDA_2 | Kept for now |
| Freshness (days since first seen) | Keep for now: `first_seen`, with `posted` alongside | How large the late-found tail is, and whether late-found listings close sooner, are checked in EDA_SDA_2 | Kept for now |
| Pay disclosed | Keep for now, alongside pay level | How much pay-transparency laws and hourly vs. salaried jobs drive it is checked in EDA_SDA_2; also lets pay level be used without dropping listings | Kept for now |
| Pay level | Keep | A direct measure of whether a listing is worth an application | Decided |
| Description length; urgency wording | Keep until EDA_SDA_3 | Length may not be intent; urgent listings stay open longer, which needs explaining | Kept for now |
| Remote mentions | Keep | Differs widely between groups (2–35%) and matters to applicants | Decided |
| Occupation group | Keep as its own feature | Large differences between groups; supports showing related jobs in the app | Decided |
| Reposts | **Keep as a feature:** `is_repost`, `times_listed`, `days_since_first_listed`, `days_since_repost_last_listed` | A job posted again and again may be hard to fill. Built from earlier copies only, so known when a listing appears | Decided |
| Hiring difficulty | Keep: age so far, `times_listed`, `days_since_repost_last_listed` | A role open for long or relisted may be hard to fill; EDA_SDA_2 first separates real difficulty from ghost listings (done in 3.1: likely ghosts are hidden, other listings open 90+ days are labelled, and "possibly hard to fill" was dropped) | Kept for now |
| Career-change accessibility | Keep: `entry_seniority`, `years_required`, `low_experience_ask`, `entry_wording`, `changer_wording`, `degree_required`, `equivalent_experience`, `career_open` | Core to the project: hard blockers **hide** a listing; softer signals raise it. From seniority and listing text, so known when a listing appears (no leakage). Keyword rules, checked in EDA_SDA_3 | Decided; rules validated in EDA_SDA_3 |
| Transferable fit | Keep for later | Needs career-changer profiles (`Profiles.ipynb`) and O*NET skills | Kept, not yet applicable |

**Data-quality fixes are not scope decisions:** no rows were removed in cleaning; 5,454 placeholder `posted` dates and 14 salary outliers were blanked, and labels were unified (cleaning report in 6.7).

### 10a Checks before the final filters

Two quick checks that can change which rows the final table keeps. Run them before section 11.
- **Missing country vs. remote:** remote listings often have no location. If the listings with no country are mostly remote, dropping them removes much of a feature we're keeping.
- **Missing language vs. missing description:** PredictLeads probably detects language from the description. If the unlabeled listings are mostly the ones without a description, dropping them is really a "has description" filter.

In [ ]:
# 10a Check: do listings with no country mention remote work more often?
remote_contract = (has_label(jobs.contract_bits, labels["contract_types"], "remote") if "remote" in labels["contract_types"]
                   else pd.Series(False, index=jobs.index))
show("Remote mentions, by whether the country is known", pd.DataFrame({
    "country_known": jobs.country.notna(), "mentions_remote": jobs.mentions_remote, "remote_contract_type": remote_contract,
}).groupby("country_known").agg(
    listings=("mentions_remote", "size"),
    mentions_remote=("mentions_remote", "mean"),
    remote_contract_type=("remote_contract_type", "mean"),
).round(3))

In [ ]:
# 10a Check: are listings with no language label mostly the ones without a description?
show("Language label vs. description (listings)",
     pd.crosstab(jobs.language.notna().map({True: "language known", False: "no language label"}),
                 jobs.has_desc.map({True: "has description", False: "no description"}), margins=True))

**What the checks show**
- **Listings without a country mention remote work twice as often.** Among listings with a description, 28.3% of the 16,778 without a country mention remote vs. 12.6% of those with one, and 11.5% vs. 7.7% use the "remote" contract type. (An earlier version showed 8.6% vs. 11.0%, because listings without a description were counted as "no".) **Decision:** keep them in this table, flagged `country_missing`. EDA_SDA_2 keeps those whose company is US-based or that mention remote work (`country_inferred`), and drops the rest.
- **Missing language almost always means no description:** 9,214 of the 9,228 (99.8%). So the language filter also removes about a quarter of the listings without descriptions (9,214 of 35,490), which is why descriptions rise from 82% to 87% in the final table. **Decision:** keep the drop, and record that it doubles as a partial "has a description" filter.

## 11. Final filters: building the EDA_SDA_1 final table

Applies every decision from section 10, all at once. Nothing earlier filters the data. The table is saved to `data/clean/analysis_table.parquet` for EDA_SDA_2 (read a few columns at a time), with the text and list columns added back from the stream's rows. 11d previews the first rows (clear before committing); 11e lists every column with its role and meaning.

In [ ]:
# 11a Settings from the decision table in section 10 (occupation groups are chosen in EDA_SDA_2)
FILTERS = {
    "language": "en",                # English only; listings with no language label are dropped
    "country": "United States",      # listings with no country are kept, flagged country_missing; EDA_SDA_2 resolves them
    "first_month": "2026-04",        # the 2-day partial March is kept, flagged partial_month, and left out of per-month counts
}

In [ ]:
# 11b Apply the filters one at a time, recording what each removes
# Counted on jobs (every cleaned listing, a few columns); the full rows are read back from disk, filtered as they load.
del jo
gc.collect()
english = jobs.language.eq(FILTERS["language"])
us_or_none = jobs.country.eq(FILTERS["country"]) | jobs.country.isna()
funnel = [("listings in the full file", stream["records"]),
          ("English or unlabeled, US or no country (the stream, 6.1)", stream["rows"]),
          ("cleaned listings, reposts flagged (6.7)", len(jobs)),
          ("English only", english.sum()),
          ("United States, or no country (flagged)", (english & us_or_none).sum())]
import pyarrow.compute as pc
import pyarrow.dataset as ds
keep = (pc.field("language") == FILTERS["language"]) & (
    (pc.field("country") == FILTERS["country"]) | pc.field("country").is_null())
final = ds.dataset(CLEANED_PATH).to_table(filter=keep).to_pandas()
final["country_missing"] = final.country.isna()
final["partial_month"] = final.month < pd.Period(FILTERS["first_month"], "M")  # hide, never remove
funnel.append((f"first seen before {FILTERS['first_month']}: kept, flagged partial_month ({final.partial_month.sum():,})", len(final)))

funnel = pd.DataFrame(funnel, columns=["step", "listings"]).set_index("step")
funnel["removed"] = (-funnel.listings.diff()).fillna(0).astype(int)
funnel["share of start"] = (funnel.listings / funnel.listings.iloc[0]).round(3)
show("Final table: listings after each filter", funnel)

In [ ]:
# 11c Save EDA_SDA_1's final table for EDA_SDA_2: the columns above, plus the text and list columns from the stream's
# rows (id, title, title_norm, job_title, desc_hash, contract_types with unified labels, tags, categories, and the raw
# fields kept for EDA_SDA_2's feature screening), added part by part. EDA_SDA_2 reads only the columns it needs.
FINAL_PATH = CLEAN_DIR / "analysis_table.parquet"
start = time.time()
save_final(final, FINAL_PATH)
show("Final analysis table at a glance", pd.DataFrame({"value": {
    "listings": len(final),
    "companies": final.company_id.nunique(),
    "occupation groups": final.occupation_group.nunique(),
    "full months": final.loc[~final.partial_month, "month"].nunique(),
    "late-March listings (partial_month)": final.partial_month.sum(),
    "still open at the snapshot": final.live.sum(),
    "columns saved": len(pq.read_schema(FINAL_PATH).names),
}}))
print(f"saved {FINAL_PATH.name} in {time.time() - start:.0f}s ({FINAL_PATH.stat().st_size / 1e9:.2f} GB)")

In [ ]:
# 11d Preview: the first rows of the final table
# Shows real listing rows. Tagged "data-rows": clear this output before committing or exporting the notebook.
final.head()

In [ ]:
# 11e Feature dictionary: every column in the final table, its role and what it means
FEATURES = {
    # column: (role, prioritization factor, description)
    "id": ("identifier", "", "PredictLeads listing id"),
    "company_id": ("identifier", "", "PredictLeads company id; links to the companies table"),
    "title": ("text", "employer intent", "Listing title as posted"),
    "row": ("identifier", "", "Row position in the stream's rows table (data/flat/all_fields_rows_full/), for joins back to it"),
    "title_norm": ("helper", "", "Lowercased title with whitespace collapsed; used to match reposts"),
    "job_title": ("text", "transferable fit", "PredictLeads' normalized title; where missing, the posted title lowercased and trimmed"),
    "job_title_fallback": ("helper", "", "job_title is the posted title (PredictLeads gave no normalized title)"),
    "has_title": ("helper", "", "The listing has a title"),
    "repost_fp": ("helper", "", "64-bit fingerprint of company, normalized title and description fingerprint; matches reposts (6.4)"),
    "contract_bits": ("helper", "", "contract_types as bit flags (labels unified), for counts in memory"),
    "category_bits": ("helper", "", "categories as bit flags, for counts in memory"),
    "n_tags": ("category", "transferable fit", "Number of tags"),
    "n_categories": ("category", "transferable fit", "Number of categories"),
    "has_recruiter": ("metadata", "", "A recruiter is named on the listing (a flag only: names and contacts are personal data and aren't kept)"),
    "last_processed": ("time", "", "When PredictLeads last processed the listing; likely moves with last_seen, so audited for leakage before any use"),
    "salary_currency": ("factor", "pay", "Currency the pay was posted in"),
    "salary_low": ("factor", "pay", "Low end of the pay range as posted, in salary_currency"),
    "salary_high": ("factor", "pay", "High end of the pay range as posted, in salary_currency"),
    "was_translated": ("metadata", "", "PredictLeads translated the title"),
    "first_seen": ("time", "freshness", "When the crawler first found the listing (UTC)"),
    "last_seen": ("time", "still open / apply-by", "When the crawler last saw the listing (UTC)"),
    "posted": ("time", "freshness", "Employer's posting date, where given (about 41%); midnight Central European time"),
    "month": ("time", "", "Month the listing was first seen; the monthly time horizon"),
    "partial_month": ("scope", "", "First seen on 2026-03-30 or 31, the data's first two days: kept, but left out of per-month counts"),
    "days_open": ("target", "still open / apply-by", "Days from first seen to last seen"),
    "live": ("target", "still open / apply-by", "Last seen within 2 days of the snapshot: still open, so its lifespan is censored"),
    "status": ("metadata", "", "PredictLeads status; only 'closed' or missing, so not used as a live flag"),
    "seniority": ("category", "career-change accessibility", "Seniority level, e.g. junior, mid_senior, manager"),
    "occupation_group": ("category", "occupation group", "O*NET occupation group (PredictLeads field onet_data.family)"),
    "occupation": ("category", "transferable fit", "O*NET occupation name; finer than the occupation group"),
    "onet_code": ("category", "occupation group", "O*NET-SOC code; maps to a role bucket in the computer and mathematical scope (EDA_SDA_2 3.6)"),
    "categories": ("category", "transferable fit", "PredictLeads category labels (27 values)"),
    "tags": ("category", "transferable fit", "PredictLeads tags; sparse (about half empty)"),
    "contract_types": ("category", "transferable fit", "Contract types, e.g. full time, part time, remote; labels unified in 6.2"),
    "language": ("scope", "", "Listing language; the final table is English only"),
    "country": ("scope", "", "Country of the first listed location; the final table is United States only"),
    "n_locations": ("category", "", "Number of locations the listing names"),
    "country_missing": ("scope", "", "No country on the listing; EDA_SDA_2 keeps it if the company is US-based or the listing mentions remote, and drops the rest"),
    "source_type": ("factor", "source", "Hiring platform, job board, or company site / other (from the url domain)"),
    "platform": ("factor", "source", "Named hiring platform or job board; empty for company sites"),
    "salary_low_usd": ("factor", "pay", "Low end of the salary range, USD, in its original time unit"),
    "salary_high_usd": ("factor", "pay", "High end of the salary range, USD, in its original time unit"),
    "salary_unit": ("factor", "pay", "Salary time unit: hour, day, week, month or year"),
    "salary_hourly_usd": ("factor", "pay", "Midpoint pay as an hourly rate (salaried pay ÷ 2,080 hours, full time); blank if not disclosed or implausible"),
    "pay_suspect": ("helper", "pay", "Pay was fixed in 6.5: reversed range swapped, or a clearly wrong unit re-read"),
    "has_desc": ("text", "employer intent", "Has a description"),
    "has_html": ("helper", "", "Description contained HTML before cleaning"),
    "desc_words": ("text", "employer intent", "Words in the cleaned description"),
    "desc_hash": ("helper", "", "Fingerprint of the cleaned description; used to match reposts"),
    "mentions_remote": ("text", "employer intent", "Title or description mentions remote work; blank if there's no description and the title doesn't say remote"),
    "urgency": ("text", "employer intent", "Description uses urgency wording (urgent, immediately, ASAP); blank without a description"),
    "years_required": ("text", "career-change accessibility", "Most years of experience the description asks for (lower end of each range, up to 20); blank if none stated"),
    "entry_wording": ("text", "career-change accessibility", "Says entry level, no experience required, will train, paid training or apprenticeship; blank without a description"),
    "changer_wording": ("text", "career-change accessibility", "Mentions career changers, transferable skills or non-traditional backgrounds; blank without a description"),
    "degree_required": ("text", "career-change accessibility", "Says a bachelor's, master's or associate degree is required; blank without a description"),
    "equivalent_experience": ("text", "career-change accessibility", "Accepts equivalent experience in place of a degree; blank without a description"),
    "entry_seniority": ("factor", "career-change accessibility", "Seniority label is entry level (junior)"),
    "low_experience_ask": ("factor", "career-change accessibility", "Asks for 2 years of experience or fewer"),
    "career_open": ("factor", "career-change accessibility", "Any sign of being open to career changers: entry-level seniority, 2 years or fewer, entry-level or training wording, or career-changer wording. Blank when nothing says yes and the text is missing. First pass; vetted in EDA_SDA_4"),
    "is_repost": ("factor", "reposts", "This job was listed before"),
    "times_listed": ("factor", "reposts", "How many times this job has been listed so far (1 = first listing)"),
    "days_since_first_listed": ("factor", "reposts", "Days since the job's original listing"),
    "days_since_repost_last_listed": ("factor", "reposts", "Days since the job's previous listing; 0 for the first listing"),
}
def saved_column(col):
    """dtype and non-null % of a saved column: from memory if loaded, else read from the file one column at a time."""
    if col in final.columns:
        return str(final[col].dtype), round(100 * final[col].notna().mean(), 1)
    c = pq.read_table(FINAL_PATH, columns=[col]).column(0)
    return str(c.type), round(100 * (1 - c.null_count / max(len(c), 1)), 1)

saved = pq.read_schema(FINAL_PATH).names
dictionary = pd.DataFrame(
    [(col, *FEATURES.get(col, ("?", "", "no description yet")), *saved_column(col)) for col in saved],
    columns=["column", "role", "prioritization factor", "description", "dtype", "non-null %"],
).set_index("column")
show("Final table: feature dictionary", dictionary)
undocumented = [c for c in saved if c not in FEATURES]
if undocumented:
    print("columns without a description:", undocumented)

## 12. Simple SDA: summary statistics of the final table

A short statistical summary of the final table: its size, how its key measures are distributed, and how it splits by month, occupation group and source. Aggregates only.

In [ ]:
# 12a Key measures: distributions
show("Final table: key measures", final[["days_open", "desc_words", "salary_hourly_usd"]]
     .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T.round(1))

In [ ]:
# 12b Key flags: share of listings
show("Final table: share of listings with each flag (text flags: among listings with a description)", pd.DataFrame({"share": {
    "closed by the snapshot": (~final.live).mean(),
    "still open at the snapshot": final.live.mean(),
    "has a description": final.has_desc.mean(),
    "salary disclosed": final.salary_hourly_usd.notna().mean(),
    "mentions remote": final.mentions_remote.mean(),
    "urgency wording": final.urgency.mean(),
    "repost (listed before)": final.is_repost.mean(),
    "entry-level seniority": final.entry_seniority.mean(),
    "asks 2 years or fewer": final.low_experience_ask.mean(),
    "entry-level or training wording": final.entry_wording.mean(),
    "career-changer wording": final.changer_wording.mean(),
    "degree required": final.degree_required.mean(),
    "open to career changers (any sign)": final.career_open.mean(),
}}).round(3))

In [ ]:
# 12c How the final table splits: by month, occupation group and source
by_month = final.loc[~final.partial_month, "month"].value_counts().sort_index()  # full months only
show("Final table: listings by month (full months; late March is flagged partial_month)",
     pd.DataFrame({"listings": by_month, "share": (by_month / by_month.sum()).round(3)}))
by_group = final.occupation_group.value_counts()
by_group = by_group[by_group > 0].head(TOP)
show("Final table: largest occupation groups", pd.DataFrame({"listings": by_group, "share": (by_group / len(final)).round(3)}))
by_source = final.source_type.value_counts()
show("Final table: source type", pd.DataFrame({"listings": by_source, "share": (by_source / len(final)).round(3)}))

In [ ]:
# 12d Chart: listings by month in the final table
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar([str(m) for m in by_month.index], by_month.values, color=BLUE, width=0.6)
ax.set(title="Final table: listings by month first seen", ylabel="listings")
ax.grid(axis="x", visible=False)
finish()

**What the data shows**
- **184,889 listings** (92.4% of the 200,000) from 47,252 companies. The filters removed 15,111 for language; the 2,522 from late March are kept, flagged `partial_month`, and listings with no country are kept, flagged. **41,984 (22.7%) are still open.**
- **A median 18.2 days open;** 10% stay open longer than 79 days: the long-open listings Part 2 examines.
- **Coverage:** 87.1% have a description and 44.8% a salary (median about $29.10 an hour; middle half $20.00–$50.00).
- **Mix:** about 35,000 listings in each of April and May, 25,500 in June, 27,400–30,300 in July–September, plus 2,522 in late March; computer and mathematical is 7.6%; 58.4% company site or other; reposts 2.3%.
- **Career-change flags:** 30.1% of the final table is entry level, 12.7% asks for 2 years or fewer, and 47.3% shows any sign of being open to career changers. Among listings with a description, 10.0% use entry-level or training wording, 3.7% require a degree and 0.4% use career-changer wording.

## 13. Final conclusion

Part 1 loaded and cleaned 200,000 listings (no rows removed), measured every candidate prioritization factor, checked which time information holds up, and compared segments for career changers. The main result: what makes a listing urgent is mostly the listing itself (employer hiring volume, source, pay disclosure), not the job title, and the best-paid field, computer and mathematical, is the hardest to enter. Every decision is defended in section 10 and applied in section 11.

**Handoff:** the final table (184,889 English listings, April–September 2026 plus late March flagged `partial_month`, including flagged listings with no country, with every group, seniority level and candidate factor) is saved for **EDA_SDA_2**, which cleans it further (2b), narrows it to the computer and mathematical scope by role bucket, and builds the **compass's 2026 side** from it: each role's openings, pay and closing speed, mapped from `onet_code`. Findings at a glance are in the executive summary; details in section 9. **EDA_SDA_2 (done)** saved the scope table: 13,887 listings in eight role buckets, and found that collection effects (crawl timing, job boards stopping after May, listings with no description) explain much of the raw gaps here.